<a href="https://colab.research.google.com/github/dataguirre/curso-ia-ciencia-de-datos/blob/main/workshops/04-workshop-rag-solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IA para ciencia de datos: Workshop 4 (soluciones)

En el Workshop 3 le pegamos el manual completo de la empresa de transporte al prompt y funciono. Pero cerramos con un problema: **eso no escala**. Con cientos de documentos no caben en la ventana de contexto, pagas por reenviarlos en cada pregunta y el modelo se distrae con tanto texto irrelevante.

En este workshop construimos la solucion: un sistema **RAG** (*Retrieval-Augmented Generation*). La idea es simple: antes de preguntarle al LLM, **buscamos** los fragmentos de los documentos que sirven para esa pregunta y le mandamos solo esos.

```
INDEXAR  (una sola vez): documentos -> fragmentos -> embeddings -> indice
PREGUNTAR  (cada vez):   pregunta -> embedding -> buscar los k mas parecidos -> prompt con esos fragmentos -> LLM -> respuesta + fuentes
```

Seguimos con **Transportes del Llano**, que ahora nos entrego no uno sino siete documentos: manual operativo, politicas de seguridad vial, mantenimiento y viaticos, reglamento interno, protocolo de emergencias y tarifas.

1. **Fragmentar**: partir los documentos en pedazos manejables.
2. **Recuperar**: buscar los pedazos relevantes, por palabras y por significado.
3. **Generar**: responder con base en lo recuperado, citando fuentes.
4. **Evaluar**: medir si el buscador encuentra lo que debe encontrar.
5. **Base de datos vectorial**: guardar el indice en ChromaDB.
6. **Interfaz**: ponerle una cara web con Gradio para que lo use alguien que no programa.

> **Antes de empezar:** `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion` y selecciona **GPU (T4)**. No es obligatoria (todo corre en CPU), pero los embeddings se calculan mas rapido.

### Configuracion inicial

Instalamos las librerias del workshop y montamos `preguntar_groq`, igual que en el Workshop 3:

- `groq`: el LLM que genera las respuestas (Workshop 2).
- `rank_bm25`: busqueda por palabras clave.
- `sentence-transformers`: el modelo de embeddings.
- `chromadb`: la base de datos vectorial.
- `gradio`: la interfaz web.

> Si Colab te pide reiniciar el entorno despues de instalar, reinicia y vuelve a ejecutar esta celda.

In [1]:
!pip install -q groq rank_bm25 "sentence-transformers>=3" "chromadb>=1.0" "gradio>=6,<7"

import os
import numpy as np
from groq import Groq
from google.colab import userdata

MODELO_GROQ = "openai/gpt-oss-20b"
client = Groq(api_key=userdata.get("GROQ_API_KEY"))


def preguntar_groq(prompt: str, system_prompt: str = None, modelo: str = MODELO_GROQ, temperature: float = 0.7) -> str:
  """Envia un prompt (y opcionalmente un system prompt) a un LLM de Groq y devuelve el texto de la respuesta."""

  mensajes = []
  if system_prompt:
    mensajes.append({"role": "system", "content": system_prompt})
  mensajes.append({"role": "user", "content": prompt})

  respuesta = client.chat.completions.create(
      model=modelo,
      messages=mensajes,
      temperature=temperature,
      reasoning_effort="low",
  )

  return respuesta.choices[0].message.content

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

## Actividad 1: Preparar los documentos

### Objetivo

Un RAG no busca en documentos completos sino en **fragmentos** (*chunks*). Hay tres razones:

1. **Precision del buscador**: un documento de 8 paginas habla de muchos temas, y su embedding termina siendo un "promedio" de todos. Un parrafo habla de una sola cosa.
2. **Economia del prompt**: el LLM solo necesita el parrafo que responde, no las 8 paginas.
3. **Limite del modelo de embeddings**: el que usaremos lee maximo 512 tokens; lo que pase de ahi lo ignora **sin avisar**.

#### Antes de empezar: sube los documentos

Los documentos estan en el drive del curso, en `rag_transportes_del_llano.zip`. Subelo a Colab (icono de carpeta en el panel izquierdo, boton de subir) **sin descomprimirlo**. Trae dos carpetas:

- `documentos/`: los 7 documentos que vamos a indexar.
- `para_probar_interfaz/`: un documento extra que usaremos en la Actividad 6. **No lo indexes todavia.**

In [2]:
import zipfile
import glob

RUTA_ZIP = "rag_transportes_del_llano.zip"

if not os.path.exists(RUTA_ZIP):
  print(f"✗ No encuentro '{RUTA_ZIP}'.")
  print("    Subelo a Colab desde el panel de la izquierda (icono de carpeta).")
else:
  with zipfile.ZipFile(RUTA_ZIP) as z:
    z.extractall(".")

  documentos = []
  for ruta in sorted(glob.glob("documentos/*.txt")):
    with open(ruta, "r", encoding="utf-8") as f:
      documentos.append({"fuente": os.path.basename(ruta), "texto": f.read()})

  total = sum(len(d["texto"].split()) for d in documentos)
  print(f"✓ {len(documentos)} documentos cargados, {total:,} palabras en total\n")
  for d in documentos:
    print(f"  {d['fuente']:<30} {len(d['texto'].split()):>5} palabras")

✓ 7 documentos cargados, 3,115 palabras en total

  manual_operativo.txt             726 palabras
  politica_mantenimiento.txt       377 palabras
  politica_seguridad_vial.txt      469 palabras
  politica_viaticos.txt            359 palabras
  protocolo_emergencias.txt        468 palabras
  reglamento_interno.txt           399 palabras
  tarifas_servicio.txt             317 palabras


#### Tarea 1: Construir la funcion `fragmentar`

Partimos cada texto en ventanas de `tamano` palabras. Para no cortar una idea justo por la mitad, las ventanas se **solapan**: cada fragmento repite las ultimas `solapamiento` palabras del anterior. Asi, cualquier frase de hasta `solapamiento` palabras aparece completa en al menos un fragmento.

### Requisitos

1. Si `solapamiento` es negativo o no es menor que `tamano`, lanzar `ValueError` (no habria forma de avanzar).
2. Partir el texto en palabras con `texto.split()` (asi los saltos de linea y espacios de mas no importan).
3. Cada fragmento tiene como maximo `tamano` palabras, unidas con un espacio.
4. Cada fragmento empieza `paso = tamano - solapamiento` palabras despues del anterior.
5. **Parar** en cuanto un fragmento llegue al final del texto (el ultimo puede quedar mas corto).
6. Si el texto esta vacio, devolver `[]`.

### Ejemplo

```
fragmentar("a b c d e f g h i j", tamano=4, solapamiento=1)
# paso = 4 - 1 = 3  ->  los fragmentos empiezan en las palabras 0, 3 y 6
== ["a b c d", "d e f g", "g h i j"]
```

El tercer fragmento ya llega a la ultima palabra, asi que ahi se para: no se genera un cuarto fragmento `"j"`.

In [3]:
def fragmentar(texto: str, tamano: int = 120, solapamiento: int = 30) -> list:
  """Parte un texto en fragmentos de maximo `tamano` palabras que se solapan `solapamiento` palabras."""

  # TODO: lanzar ValueError si el solapamiento es negativo o no es menor que el tamano
  if solapamiento < 0 or solapamiento >= tamano:
    raise ValueError("El solapamiento debe estar entre 0 y tamano - 1")

  # TODO: partir el texto en palabras y calcular el paso entre fragmentos
  palabras = texto.split()
  paso = tamano - solapamiento

  fragmentos = []
  # TODO: recorrer las palabras de `paso` en `paso`, armar cada fragmento con " ".join(...)
  #       y parar cuando un fragmento llegue al final del texto
  for inicio in range(0, len(palabras), paso):
    fragmentos.append(" ".join(palabras[inicio:inicio + tamano]))
    if inicio + tamano >= len(palabras):
      break

  return fragmentos

**Evaluacion de implementacion**

In [4]:
# @title
# Celda de validacion. No modificar.

def _validar_fragmentar():
  if "fragmentar" not in globals():
    print("✗ Todavia no existe 'fragmentar'. Ejecuta la celda anterior.")
    return

  fallas = []

  def revisar(descripcion, calcular, esperado):
    try:
      obtenido = calcular()
    except Exception as e:
      obtenido = f"{type(e).__name__}: {e}"
    if obtenido == esperado:
      print(f"✓ {descripcion}")
    else:
      print(f"✗ {descripcion}\n    esperaba: {esperado}\n    obtuvo:   {obtenido}")
      fallas.append(descripcion)

  diez = "a b c d e f g h i j"
  once = "a b c d e f g h i j k"

  revisar("ejemplo del enunciado (tamano=4, solapamiento=1)",
          lambda: fragmentar(diez, 4, 1), ["a b c d", "d e f g", "g h i j"])
  revisar("si sobran palabras, el ultimo fragmento queda mas corto",
          lambda: fragmentar(once, 4, 1), ["a b c d", "d e f g", "g h i j", "j k"])
  revisar("sin solapamiento, ningun fragmento repite palabras",
          lambda: fragmentar(diez, 5, 0), ["a b c d e", "f g h i j"])
  revisar("texto mas corto que el tamano -> un solo fragmento",
          lambda: fragmentar("hola mundo", 4, 1), ["hola mundo"])
  revisar("texto vacio -> lista vacia",
          lambda: fragmentar("", 4, 1), [])
  revisar("saltos de linea y espacios de mas no importan",
          lambda: fragmentar("a  b\nc\n\nd e", 3, 1), ["a b c", "c d e"])

  for tam, sol, nota in [(4, 4, "solapamiento igual al tamano"),
                         (4, 6, "solapamiento mayor que el tamano"),
                         (4, -1, "solapamiento negativo")]:
    try:
      fragmentar(diez, tam, sol)
      print(f"✗ {nota}: debia lanzar ValueError y no lanzo nada")
      fallas.append(nota)
    except ValueError:
      print(f"✓ {nota} -> ValueError")
    except Exception as e:
      print(f"✗ {nota}: debia lanzar ValueError, lanzo {type(e).__name__}: {e}")
      fallas.append(nota)

  # Propiedades sobre un texto largo: 1.000 palabras, tamano 120, solapamiento 30
  largo = " ".join(f"p{i}" for i in range(1000))
  try:
    frags = fragmentar(largo, 120, 30)
  except Exception as e:
    print(f"✗ fragmentar(texto de 1.000 palabras) lanzo {type(e).__name__}: {e}")
    fallas.append("texto largo")
    frags = None

  if frags is not None:
    revisar("texto largo: ningun fragmento pasa de 120 palabras",
            lambda: all(len(f.split()) <= 120 for f in frags), True)
    revisar("texto largo: cada fragmento empieza con las 30 ultimas palabras del anterior",
            lambda: all(a.split()[-30:] == b.split()[:30] for a, b in zip(frags, frags[1:])), True)
    revisar("texto largo: el primer fragmento empieza en p0 y el ultimo termina en p999",
            lambda: (frags[0].split()[0], frags[-1].split()[-1]), ("p0", "p999"))
    revisar("texto largo: 11 fragmentos (empiezan en 0, 90, 180, ..., 900)",
            lambda: len(frags), 11)

  print()
  if fallas:
    print(f"{len(fallas)} problema(s) por corregir.")
  else:
    print("\U0001F389 ¡Todo funciona correctamente!")


_validar_fragmentar()

✓ ejemplo del enunciado (tamano=4, solapamiento=1)
✓ si sobran palabras, el ultimo fragmento queda mas corto
✓ sin solapamiento, ningun fragmento repite palabras
✓ texto mas corto que el tamano -> un solo fragmento
✓ texto vacio -> lista vacia
✓ saltos de linea y espacios de mas no importan
✓ solapamiento igual al tamano -> ValueError
✓ solapamiento mayor que el tamano -> ValueError
✓ solapamiento negativo -> ValueError
✓ texto largo: ningun fragmento pasa de 120 palabras
✓ texto largo: cada fragmento empieza con las 30 ultimas palabras del anterior
✓ texto largo: el primer fragmento empieza en p0 y el ultimo termina en p999
✓ texto largo: 11 fragmentos (empiezan en 0, 90, 180, ..., 900)

🎉 ¡Todo funciona correctamente!


#### Tarea 2: Fragmentar todo el corpus

Cada fragmento tiene que **recordar de que documento viene**: es lo que despues nos va a permitir citar fuentes. Implementa `fragmentar_corpus`, que recibe la lista `documentos` y devuelve una lista de diccionarios:

```
{"id": "politica_viaticos.txt#0", "fuente": "politica_viaticos.txt", "texto": "..."}
```

### Requisitos

1. Fragmentar el texto de cada documento con `fragmentar`, pasandole el `tamano` y el `solapamiento` recibidos.
2. El `id` es la fuente, un `#` y el numero del fragmento **dentro de su documento**, empezando en 0.
3. Los fragmentos van en orden: todos los del primer documento, luego los del segundo, etc.

In [5]:
def fragmentar_corpus(documentos: list, tamano: int = 120, solapamiento: int = 30) -> list:
  """Fragmenta cada documento y devuelve una lista de dicts con id, fuente y texto."""

  fragmentos = []

  # TODO: para cada documento, fragmentar su texto y agregar un dict por fragmento
  for doc in documentos:
    for i, texto in enumerate(fragmentar(doc["texto"], tamano, solapamiento)):
      fragmentos.append({"id": f"{doc['fuente']}#{i}", "fuente": doc["fuente"], "texto": texto})

  return fragmentos

**Evaluacion de implementacion**

In [6]:
# @title
# Celda de validacion. No modificar.

def _validar_fragmentar_corpus():
  for nombre in ("fragmentar", "fragmentar_corpus"):
    if nombre not in globals():
      print(f"✗ Todavia no existe '{nombre}'. Ejecuta las celdas anteriores.")
      return

  fallas = []

  def revisar(descripcion, calcular, esperado):
    try:
      obtenido = calcular()
    except Exception as e:
      obtenido = f"{type(e).__name__}: {e}"
    if obtenido == esperado:
      print(f"✓ {descripcion}")
    else:
      print(f"✗ {descripcion}\n    esperaba: {esperado}\n    obtuvo:   {obtenido}")
      fallas.append(descripcion)

  docs = [
      {"fuente": "a.txt", "texto": "uno dos tres cuatro cinco"},
      {"fuente": "b.txt", "texto": "seis siete"},
  ]

  revisar("un dict por fragmento, con id, fuente y texto",
          lambda: fragmentar_corpus(docs, 3, 1),
          [{"id": "a.txt#0", "fuente": "a.txt", "texto": "uno dos tres"},
           {"id": "a.txt#1", "fuente": "a.txt", "texto": "tres cuatro cinco"},
           {"id": "b.txt#0", "fuente": "b.txt", "texto": "seis siete"}])
  revisar("respeta el tamano y el solapamiento que recibe",
          lambda: [f["texto"] for f in fragmentar_corpus(docs, 2, 0)],
          ["uno dos", "tres cuatro", "cinco", "seis siete"])
  revisar("la numeracion vuelve a 0 en cada documento",
          lambda: [f["id"] for f in fragmentar_corpus(docs, 2, 0)],
          ["a.txt#0", "a.txt#1", "a.txt#2", "b.txt#0"])
  revisar("sin documentos -> lista vacia",
          lambda: fragmentar_corpus([], 3, 1), [])

  if "documentos" in globals():
    revisar("en el corpus real, todos los id son distintos",
            lambda: len({f["id"] for f in fragmentar_corpus(documentos)}) == len(fragmentar_corpus(documentos)),
            True)

  print()
  if fallas:
    print(f"{len(fallas)} problema(s) por corregir.")
  else:
    print("\U0001F389 ¡Todo funciona correctamente!")


_validar_fragmentar_corpus()

✓ un dict por fragmento, con id, fuente y texto
✓ respeta el tamano y el solapamiento que recibe
✓ la numeracion vuelve a 0 en cada documento
✓ sin documentos -> lista vacia
✓ en el corpus real, todos los id son distintos

🎉 ¡Todo funciona correctamente!


#### Tarea 3: Fragmentar el corpus real

Esta celda ya esta completa. Usamos fragmentos de 120 palabras con 30 de solapamiento; en la Actividad 4 veremos que pasa con otros tamanos.

In [7]:
import textwrap

fragmentos = fragmentar_corpus(documentos, tamano=120, solapamiento=30)

print(f"{len(documentos)} documentos -> {len(fragmentos)} fragmentos\n")

ejemplo = next(f for f in fragmentos if f["id"] == "politica_viaticos.txt#1")
print(f"id:     {ejemplo['id']}")
print(f"fuente: {ejemplo['fuente']}")
print("texto:")
print(textwrap.fill(ejemplo["texto"], width=100, initial_indent="  ", subsequent_indent="  "))

7 documentos -> 35 fragmentos

id:     politica_viaticos.txt#1
fuente: politica_viaticos.txt
texto:
  del Meta y el Casanare, por ejemplo hacia Bogotá, Tunja o la Costa, el viático es de 85.000 pesos
  por día de viaje. Un día de viaje se cuenta cuando el conductor pasa más de 8 horas fuera de su
  base. 3. HOSPEDAJE Cuando el viaje exige pasar la noche fuera de la base, la empresa reconoce el
  hospedaje hasta un tope de 120.000 pesos por noche, siempre que el conductor presente factura
  electrónica a nombre de la empresa. El conductor también puede dormir en la cabina si el vehículo
  tiene camarote y está en un parqueadero vigilado; en ese caso recibe un auxilio de 40.000 pesos
  por noche. 4. PEAJES Y COMBUSTIBLE Los peajes se pagan con el tag


Fijate que el fragmento empieza y termina a mitad de frase. Fragmentar por numero de palabras es sencillo, pero es **ciego a la estructura** del documento: no sabe donde empieza una seccion. En las extensiones proponemos fragmentar por secciones.

## Actividad 2: Recuperar

### Objetivo

Dada una pregunta, encontrar los `k` fragmentos mas utiles para responderla. Veremos dos formas:

1. **Por palabras** (busqueda lexica): gana el fragmento que comparte mas palabras con la pregunta. En el fondo es **contar**, como los n-gramas del Workshop 1.
2. **Por significado** (busqueda semantica): convertimos la pregunta y los fragmentos en vectores (**embeddings**) y buscamos los mas cercanos.

#### Tarea 1: Busqueda por palabras clave con BM25

Esta celda ya esta completa: solo ejecutala. **BM25** es el algoritmo clasico de los buscadores. Le da puntaje a cada fragmento segun:

- cuantas veces aparecen en el las palabras de la pregunta,
- que tan **raras** son esas palabras en el corpus (una palabra que esta en todos los fragmentos, como "la", casi no suma),
- y penaliza un poco los fragmentos largos, que tienen mas palabras "por suerte".

El `tokenizar` es primo del Workshop 1, con una diferencia: tambien quita las tildes (`vehículo` -> `vehiculo`), porque los documentos las tienen y las preguntas de los usuarios muchas veces no.

In [8]:
import re
import unicodedata
from rank_bm25 import BM25Okapi


def normalizar(texto: str) -> str:
  """Minusculas y sin tildes: 'Vehículo' -> 'vehiculo'."""
  texto = unicodedata.normalize("NFD", texto.lower())
  return "".join(c for c in texto if unicodedata.category(c) != "Mn")


def tokenizar(texto: str) -> list:
  return re.findall(r"[a-z0-9]+", normalizar(texto))


def buscar_bm25(pregunta: str, fragmentos: list, indice, k: int = 3) -> list:
  """Devuelve los k fragmentos con mayor puntaje BM25, cada uno con su 'score'."""
  puntajes = indice.get_scores(tokenizar(pregunta))
  mejores = np.argsort(-puntajes)[:k]
  return [{**fragmentos[i], "score": float(puntajes[i])} for i in mejores]


def mostrar_resultados(pregunta: str, resultados: list, largo: int = 90):
  print(f"P: {pregunta}")
  for i, r in enumerate(resultados, start=1):
    print(f"  [{i}] {r['score']:6.2f}  {r['id']:<30} {r['texto'][:largo]}...")
  print()


indice_bm25 = BM25Okapi([tokenizar(f["texto"]) for f in fragmentos])

for pregunta in ["¿Cuanto se paga de viaticos en ruta nacional?",
                 "¿Me puedo tomar una cerveza antes de salir a ruta?",
                 "¿Que hago si me roban el camion?"]:
  mostrar_resultados(pregunta, buscar_bm25(pregunta, fragmentos, indice_bm25))

P: ¿Cuanto se paga de viaticos en ruta nacional?
  [1]  10.03  politica_viaticos.txt#0        TRANSPORTES DEL LLANO S.A.S. POLÍTICA DE VIÁTICOS Y GASTOS DE VIAJE Código: VG-004 — Vigen...
  [2]   8.31  politica_viaticos.txt#2        tiene camarote y está en un parqueadero vigilado; en ese caso recibe un auxilio de 40.000 ...
  [3]   6.20  manual_operativo.txt#7         de ruta, cierres de vía o problemas con la carga. El despachador es la única persona autor...

P: ¿Me puedo tomar una cerveza antes de salir a ruta?
  [1]   6.45  politica_mantenimiento.txt#0   TRANSPORTES DEL LLANO S.A.S. POLÍTICA DE MANTENIMIENTO DE FLOTA Código: MF-002 — Versión 3...
  [2]   5.69  manual_operativo.txt#4         momento del corte, porque aumenta su acidez. Por eso, el tiempo entre la cosecha y la entr...
  [3]   4.40  politica_mantenimiento.txt#2   asignada. 4. LLANTAS La profundidad mínima del labrado permitida es de 3 milímetros en tod...

P: ¿Que hago si me roban el camion?
  [1]   4.81  manual_oper

La primera funciona: "viaticos", "ruta" y "nacional" aparecen tal cual en la politica de viaticos. Las otras dos fallan, y por la misma razon: el documento dice **alcohol**, no "cerveza"; y dice **robo**, no "roban". Para BM25 son palabras distintas y punto.

#### Tarea 2: Embeddings

Un modelo de embeddings convierte un texto en un vector de numeros (aqui, 384) de forma que **textos con significado parecido quedan cerca**, aunque no compartan palabras.

Usamos `intfloat/multilingual-e5-small`: pequeno (118M parametros), multilingue y entrenado especificamente para busqueda. Tiene un detalle: fue entrenado con **prefijos**. A las preguntas hay que anteponerles `"query: "` y a los fragmentos `"passage: "`. Si se te olvidan, sigue funcionando, pero peor y sin avisar. (Es la misma idea de `apply_chat_template` en el Workshop 2: cada modelo espera su formato.)

La funcion `embeber` ya esta lista y se encarga de los prefijos. Ejecuta la celda.

In [9]:
from sentence_transformers import SentenceTransformer

MODELO_EMB = "intfloat/multilingual-e5-small"
modelo_emb = SentenceTransformer(MODELO_EMB)


def embeber(textos: list, tipo: str = "passage") -> np.ndarray:
  """Convierte una lista de textos en una matriz de embeddings (un vector por fila).

  tipo="query" para preguntas y tipo="passage" para fragmentos (prefijos del modelo e5).
  """
  prefijo = "query: " if tipo == "query" else "passage: "
  return modelo_emb.encode([prefijo + t for t in textos], normalize_embeddings=True)


vectores = embeber(["Hola, ¿como estas?", "El camion lleva fruto de palma"])
print("Forma:", vectores.shape, "  (2 textos, 384 numeros cada uno)")
print("Primeros 6 numeros del primer vector:", np.round(vectores[0][:6], 3))

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Forma: (2, 384)   (2 textos, 384 numeros cada uno)
Primeros 6 numeros del primer vector: [ 0.05   0.007 -0.064 -0.079  0.031 -0.066]


#### Tarea 3: Construir la funcion `similitud_coseno`

Para saber que tan cerca estan dos embeddings usamos la **similitud coseno**: el coseno del angulo entre los vectores. Vale 1 si apuntan en la misma direccion, 0 si son perpendiculares y -1 si son opuestos.

$$\cos(q, d) = \frac{q \cdot d}{\|q\| \, \|d\|}$$

### Requisitos

1. Recibe un vector `consulta` de forma `(d,)` y una `matriz` de forma `(n, d)`, con un vector por fila.
2. Devuelve un arreglo de forma `(n,)`: la similitud de la consulta con cada fila.
3. **No asumas** que los vectores estan normalizados: divide por las normas. (`embeber` si los normaliza, y en ese caso el coseno es solo el producto punto, pero tu funcion debe servir para cualquier vector.)
4. Sin ciclos `for`: usa operaciones de numpy.

### Ejemplo

```
similitud_coseno(np.array([1, 0]), np.array([[1, 0], [0, 1], [-1, 0], [3, 0]]))
== array([1., 0., -1., 1.])      # igual, perpendicular, opuesto, misma direccion
```

Pistas: `matriz @ consulta` calcula todos los productos punto de una vez; `np.linalg.norm(matriz, axis=1)` da la norma de cada fila.

In [10]:
def similitud_coseno(consulta: np.ndarray, matriz: np.ndarray) -> np.ndarray:
  """Similitud coseno entre el vector `consulta` y cada fila de `matriz`."""

  # TODO: calcular el producto punto de la consulta con cada fila de la matriz
  productos = matriz @ consulta

  # TODO: dividir entre (norma de cada fila) * (norma de la consulta) y devolver
  normas = np.linalg.norm(matriz, axis=1) * np.linalg.norm(consulta)
  return productos / normas

**Evaluacion de implementacion**

In [11]:
# @title
# Celda de validacion. No modificar.

def _validar_similitud_coseno():
  if "similitud_coseno" not in globals():
    print("✗ Todavia no existe 'similitud_coseno'. Ejecuta la celda anterior.")
    return

  fallas = []

  def revisar(descripcion, consulta, matriz, esperado):
    etiqueta = f"{descripcion}"
    try:
      obtenido = similitud_coseno(np.array(consulta, dtype=float), np.array(matriz, dtype=float))
    except Exception as e:
      print(f"✗ {etiqueta}\n    lanzo {type(e).__name__}: {e}")
      fallas.append(descripcion)
      return
    if obtenido is None:
      print(f"✗ {etiqueta}\n    devolvio None (¿te falto el return?)")
      fallas.append(descripcion)
      return
    obtenido = np.asarray(obtenido)
    if obtenido.shape != (len(matriz),):
      print(f"✗ {etiqueta}\n    debia tener forma {(len(matriz),)}, tiene {obtenido.shape}")
      fallas.append(descripcion)
    elif np.allclose(obtenido, esperado, atol=1e-6):
      print(f"✓ {etiqueta}  ->  {np.round(obtenido, 4)}")
    else:
      print(f"✗ {etiqueta}\n    esperaba: {np.round(esperado, 4)}\n    obtuvo:   {np.round(obtenido, 4)}")
      fallas.append(descripcion)

  revisar("ejemplo del enunciado: igual, perpendicular, opuesto, misma direccion",
          [1, 0], [[1, 0], [0, 1], [-1, 0], [3, 0]], [1, 0, -1, 1])
  revisar("vectores NO normalizados (divide por las normas)",
          [2, 0], [[5, 5]], [np.sqrt(2) / 2])
  revisar("tres dimensiones",
          [1, 2, 3], [[1, 2, 3], [2, 4, 6], [-1, -2, -3], [3, -3, 1]], [1, 1, -1, 0])
  revisar("una sola fila",
          [0, 1], [[0, 7]], [1])

  rng = np.random.default_rng(0)
  q = rng.normal(size=8); q /= np.linalg.norm(q)
  M = rng.normal(size=(5, 8)); M /= np.linalg.norm(M, axis=1, keepdims=True)
  revisar("con vectores normalizados coincide con el producto punto",
          q, M, M @ q)

  print()
  if fallas:
    print(f"{len(fallas)} problema(s) por corregir.")
  else:
    print("\U0001F389 ¡Todo funciona correctamente!")


_validar_similitud_coseno()

✓ ejemplo del enunciado: igual, perpendicular, opuesto, misma direccion  ->  [ 1.  0. -1.  1.]
✓ vectores NO normalizados (divide por las normas)  ->  [0.7071]
✓ tres dimensiones  ->  [ 1.  1. -1.  0.]
✓ una sola fila  ->  [1.]
✓ con vectores normalizados coincide con el producto punto  ->  [-0.2489  0.0993 -0.4735  0.7797  0.6224]

🎉 ¡Todo funciona correctamente!


Probemos el modelo de embeddings con tu funcion. La pregunta menciona "plata" y "comida"; ninguna de las frases dice "comida", y solo la del banco dice "plata".

In [12]:
pregunta = "¿Cuanta plata le dan al conductor para la comida?"
frases = [
    "Los viaticos de ruta nacional son de 85.000 pesos por dia de viaje.",
    "El mantenimiento preventivo se hace cada 10.000 kilometros.",
    "La plata del banco se guarda en la caja fuerte.",
]

sims = similitud_coseno(embeber([pregunta], tipo="query")[0], embeber(frases))

print(f"P: {pregunta}\n")
for frase, s in sorted(zip(frases, sims), key=lambda x: -x[1]):
  print(f"  {s:.3f}  {frase}")

P: ¿Cuanta plata le dan al conductor para la comida?

  0.822  Los viaticos de ruta nacional son de 85.000 pesos por dia de viaje.
  0.793  El mantenimiento preventivo se hace cada 10.000 kilometros.
  0.791  La plata del banco se guarda en la caja fuerte.


Lo esperable es que la frase de viaticos quede arriba aunque no comparte palabras clave con la pregunta, y que la del banco, que si dice "plata", no gane. Ojo con los numeros: con e5 las similitudes suelen quedar todas entre 0.7 y 0.9. **Lo que importa es el orden, no el valor absoluto.**

#### Tarea 4: Construir la funcion `buscar`

### Requisitos

1. Embeber la pregunta con `tipo="query"`. Ojo: `embeber` recibe una **lista** y devuelve una **matriz**; pasale `[pregunta]` y toma la fila `[0]`.
2. Calcular la similitud de la pregunta con `embeddings` (una fila por fragmento, en el mismo orden que `fragmentos`).
3. Quedarse con los `k` indices de mayor similitud, de mayor a menor. Pista: `np.argsort(-similitudes)[:k]`.
4. Devolver una lista de diccionarios: una **copia** de cada fragmento con una llave extra `"score"` (un `float`). Es el mismo formato que devuelve `buscar_bm25`. No modifiques los diccionarios originales: usa `{**fragmento, "score": ...}`.

In [13]:
def buscar(pregunta: str, fragmentos: list, embeddings: np.ndarray, k: int = 3) -> list:
  """Devuelve los k fragmentos mas parecidos a la pregunta, cada uno con su 'score'."""

  # TODO: embeber la pregunta (tipo="query") y calcular su similitud con cada fragmento
  vector_pregunta = embeber([pregunta], tipo="query")[0]
  similitudes = similitud_coseno(vector_pregunta, embeddings)

  # TODO: quedarse con los indices de las k similitudes mas altas, de mayor a menor
  mejores = np.argsort(-similitudes)[:k]

  # TODO: devolver una copia de cada fragmento con su "score"
  return [{**fragmentos[i], "score": float(similitudes[i])} for i in mejores]

**Evaluacion de implementacion**

Esta validacion reemplaza el modelo de embeddings por uno falso, asi que no depende de lo que haya aprendido e5.

In [14]:
# @title
# Celda de validacion. No modificar.
from unittest.mock import patch

def _validar_buscar():
  for nombre in ("similitud_coseno", "buscar", "embeber"):
    if nombre not in globals():
      print(f"✗ Todavia no existe '{nombre}'. Ejecuta las celdas anteriores.")
      return

  llamadas = []

  def _embeber_falso(textos, tipo="passage"):
    llamadas.append({"textos": textos, "tipo": tipo})
    return np.array([[1.0, 0.0]])

  frags = [
      {"id": "a#0", "fuente": "a.txt", "texto": "perpendicular"},
      {"id": "a#1", "fuente": "a.txt", "texto": "identico"},
      {"id": "b#0", "fuente": "b.txt", "texto": "diagonal"},
      {"id": "b#1", "fuente": "b.txt", "texto": "opuesto"},
  ]
  embs = np.array([[0.0, 1.0], [1.0, 0.0], [0.7, 0.7], [-1.0, 0.0]])
  # similitudes con [1, 0]:  0,  1,  0.707,  -1   ->  orden: a#1, b#0, a#0, b#1

  fallas = []
  with patch("__main__.embeber", _embeber_falso):
    try:
      r2 = buscar("mi pregunta", frags, embs, k=2)
      r4 = buscar("mi pregunta", frags, embs, k=4)
    except Exception as e:
      print(f"✗ buscar(...) lanzo {type(e).__name__}: {e}")
      return

  if not llamadas:
    print("✗ No se llamo a 'embeber' para convertir la pregunta en vector.")
    fallas.append("embeber")
  else:
    if not isinstance(llamadas[0]["textos"], list):
      print("✗ 'embeber' debe recibir una LISTA de textos: embeber([pregunta], tipo='query')")
      fallas.append("lista")
    else:
      print("✓ Le pasa a 'embeber' una lista con la pregunta")
    if llamadas[0]["tipo"] != "query":
      print(f"✗ La pregunta debe embeberse con tipo='query', se uso tipo={llamadas[0]['tipo']!r}")
      fallas.append("tipo")
    else:
      print("✓ Embebe la pregunta con tipo='query'")

  if not isinstance(r2, list) or not all(isinstance(x, dict) for x in r2):
    print(f"✗ Debia devolver una lista de diccionarios, devolvio: {r2!r}")
    print(f"\n{len(fallas) + 1} problema(s) por corregir.")
    return

  ids = [x.get("id") for x in r2]
  if ids == ["a#1", "b#0"]:
    print("✓ Devuelve los k=2 mas parecidos, de mayor a menor similitud")
  else:
    print(f"✗ Con k=2 esperaba los ids ['a#1', 'b#0'], obtuvo {ids}")
    fallas.append("orden")

  if [x.get("id") for x in r4] == ["a#1", "b#0", "a#0", "b#1"]:
    print("✓ Con k=4 devuelve todos, en orden")
  else:
    print(f"✗ Con k=4 esperaba ['a#1', 'b#0', 'a#0', 'b#1'], obtuvo {[x.get('id') for x in r4]}")
    fallas.append("k=4")

  scores = [x.get("score") for x in r2]
  if len(scores) == 2 and all(isinstance(s, float) for s in scores) and np.allclose(scores, [1.0, 0.7 / np.hypot(0.7, 0.7)], atol=1e-4):
    print(f"✓ Cada resultado trae su 'score' como float: {np.round(scores, 4).tolist()}")
  else:
    print(f"✗ Los 'score' debian ser floats ~[1.0, 0.7071], son {scores}")
    fallas.append("score")

  if all({"id", "fuente", "texto"} <= set(x) for x in r2):
    print("✓ Conserva id, fuente y texto de cada fragmento")
  else:
    print("✗ A los resultados les faltan llaves del fragmento original (id, fuente, texto)")
    fallas.append("llaves")

  if any("score" in f for f in frags):
    print("✗ Se modificaron los fragmentos originales (les quedo la llave 'score'). Usa {**fragmento, 'score': ...}")
    fallas.append("mutacion")
  else:
    print("✓ No modifica los fragmentos originales")

  print()
  if fallas:
    print(f"{len(fallas)} problema(s) por corregir.")
  else:
    print("\U0001F389 ¡Todo funciona correctamente!")


_validar_buscar()

✓ Le pasa a 'embeber' una lista con la pregunta
✓ Embebe la pregunta con tipo='query'
✓ Devuelve los k=2 mas parecidos, de mayor a menor similitud
✓ Con k=4 devuelve todos, en orden
✓ Cada resultado trae su 'score' como float: [1.0, 0.7071]
✓ Conserva id, fuente y texto de cada fragmento
✓ No modifica los fragmentos originales

🎉 ¡Todo funciona correctamente!


#### Tarea 5: Indexar el corpus y comparar los dos buscadores

Ahora si **indexamos**: embebemos todos los fragmentos **una sola vez** y guardamos la matriz. Cada pregunta nueva solo requiere embeber la pregunta.

Tambien empaquetamos cada buscador en una funcion que solo recibe `(pregunta, k)`. Asi el resto del notebook puede usar cualquiera de los dos sin saber como funciona por dentro.

In [15]:
import time

inicio = time.time()
embeddings_fragmentos = embeber([f["texto"] for f in fragmentos], tipo="passage")
print(f"✓ {len(fragmentos)} fragmentos embebidos en {time.time() - inicio:.1f}s -> matriz {embeddings_fragmentos.shape}")


def buscador_semantico(pregunta: str, k: int = 3) -> list:
  return buscar(pregunta, fragmentos, embeddings_fragmentos, k)


def buscador_bm25(pregunta: str, k: int = 3) -> list:
  return buscar_bm25(pregunta, fragmentos, indice_bm25, k)

✓ 35 fragmentos embebidos en 0.4s -> matriz (35, 384)


In [16]:
for pregunta in ["¿Cuanto se paga de viaticos en ruta nacional?",
                 "¿Me puedo tomar una cerveza antes de salir a ruta?",
                 "¿Que hago si me roban el camion?"]:
  print("=" * 110)
  print("BM25")
  mostrar_resultados(pregunta, buscador_bm25(pregunta))
  print("SEMANTICO")
  mostrar_resultados(pregunta, buscador_semantico(pregunta))

BM25
P: ¿Cuanto se paga de viaticos en ruta nacional?
  [1]  10.03  politica_viaticos.txt#0        TRANSPORTES DEL LLANO S.A.S. POLÍTICA DE VIÁTICOS Y GASTOS DE VIAJE Código: VG-004 — Vigen...
  [2]   8.31  politica_viaticos.txt#2        tiene camarote y está en un parqueadero vigilado; en ese caso recibe un auxilio de 40.000 ...
  [3]   6.20  manual_operativo.txt#7         de ruta, cierres de vía o problemas con la carga. El despachador es la única persona autor...

SEMANTICO
P: ¿Cuanto se paga de viaticos en ruta nacional?
  [1]   0.88  politica_viaticos.txt#0        TRANSPORTES DEL LLANO S.A.S. POLÍTICA DE VIÁTICOS Y GASTOS DE VIAJE Código: VG-004 — Vigen...
  [2]   0.86  politica_viaticos.txt#2        tiene camarote y está en un parqueadero vigilado; en ese caso recibe un auxilio de 40.000 ...
  [3]   0.85  politica_viaticos.txt#1        del Meta y el Casanare, por ejemplo hacia Bogotá, Tunja o la Costa, el viático es de 85.00...

BM25
P: ¿Me puedo tomar una cerveza antes de salir 

La busqueda semantica deberia encontrar la seccion de alcohol para "cerveza" y el protocolo de robo para "roban", donde BM25 fallo.

Pero no concluyas que BM25 sobra. Es mejor con **codigos, siglas, nombres propios y numeros exactos** (prueba con "¿Que es el RNDC?" o "licencia C3"), donde un embedding puede confundir cosas que "suenan parecido". En la Actividad 4 los vamos a medir en serio en vez de mirar tres ejemplos.

## Actividad 3: Generar

### Objetivo

Ya sabemos encontrar los fragmentos. Ahora completamos la "G" de RAG: armar el prompt con ellos y pedirle la respuesta al LLM. Hay dos diferencias con `responder_con_documento` del Workshop 3:

1. El prompt lleva solo `k` fragmentos, no todos los documentos.
2. Cada fragmento va **numerado y con su fuente**, para que el modelo **cite** de donde saco cada dato.

Citar es lo que vuelve **verificable** una respuesta: si el asistente dice "85.000 pesos [2]", cualquiera puede ir al fragmento 2 y comprobarlo.

#### Tarea 1: Construir la funcion `construir_prompt_rag`

### Requisitos

1. Un bloque por fragmento, numerados desde 1. La primera linea del bloque es `[n] (fuente: <fuente>)` y la segunda es el texto.
2. Los bloques se separan con una linea en blanco.
3. Al final va la linea `Pregunta: <pregunta>`, tambien separada por una linea en blanco.

### Ejemplo

Con dos fragmentos, el prompt se ve asi:

```
[1] (fuente: politica_viaticos.txt)
En ruta regional, es decir, en viajes dentro del Meta...

[2] (fuente: manual_operativo.txt)
Todo viaje inicia y termina con un reporte...

Pregunta: ¿Cuanto se paga de viaticos en ruta nacional?
```

Pista: arma una lista de bloques y unela con `"\n\n".join(bloques)`.

In [17]:
def construir_prompt_rag(pregunta: str, fragmentos: list) -> str:
  """Arma el prompt con los fragmentos numerados (y su fuente), seguidos de la pregunta."""

  bloques = []

  # TODO: agregar un bloque "[n] (fuente: ...)\n<texto>" por fragmento, numerando desde 1
  for i, fragmento in enumerate(fragmentos, start=1):
    bloques.append(f"[{i}] (fuente: {fragmento['fuente']})\n{fragmento['texto']}")

  # TODO: agregar la pregunta al final y unir todos los bloques con una linea en blanco
  bloques.append(f"Pregunta: {pregunta}")
  return "\n\n".join(bloques)

**Evaluacion de implementacion**

In [18]:
# @title
# Celda de validacion. No modificar.

def _validar_construir_prompt_rag():
  if "construir_prompt_rag" not in globals():
    print("✗ Todavia no existe 'construir_prompt_rag'. Ejecuta la celda anterior.")
    return

  frags = [
      {"id": "a.txt#0", "fuente": "a.txt", "texto": "El cielo es azul."},
      {"id": "b.txt#3", "fuente": "b.txt", "texto": "El pasto es verde."},
  ]
  esperado = (
      "[1] (fuente: a.txt)\nEl cielo es azul.\n\n"
      "[2] (fuente: b.txt)\nEl pasto es verde.\n\n"
      "Pregunta: ¿De que color es el pasto?"
  )

  try:
    obtenido = construir_prompt_rag("¿De que color es el pasto?", frags)
    solo_pregunta = construir_prompt_rag("¿Hola?", [])
  except Exception as e:
    print(f"✗ construir_prompt_rag(...) lanzo {type(e).__name__}: {e}")
    return

  if not isinstance(obtenido, str) or not obtenido:
    print(f"✗ Debia devolver un string con el prompt, devolvio {obtenido!r}")
    return

  fallas = []

  def revisar(descripcion, condicion):
    if condicion:
      print(f"✓ {descripcion}")
    else:
      print(f"✗ {descripcion}")
      fallas.append(descripcion)

  revisar("incluye el texto de cada fragmento", "El cielo es azul." in obtenido and "El pasto es verde." in obtenido)
  revisar("numera desde 1 y con la fuente: '[1] (fuente: a.txt)', '[2] (fuente: b.txt)'",
          "[1] (fuente: a.txt)" in obtenido and "[2] (fuente: b.txt)" in obtenido and "[0]" not in obtenido)
  revisar("termina con 'Pregunta: <pregunta>'", obtenido.rstrip().endswith("Pregunta: ¿De que color es el pasto?"))
  revisar("sin fragmentos, el prompt es solo la pregunta", solo_pregunta.strip() == "Pregunta: ¿Hola?")
  revisar("formato exacto (bloques separados por una linea en blanco)", obtenido == esperado)
  if obtenido != esperado:
    print(f"    esperaba:\n{esperado!r}\n    obtuvo:\n{obtenido!r}")

  print()
  if fallas:
    print(f"{len(fallas)} problema(s) por corregir.")
  else:
    print("\U0001F389 ¡Todo funciona correctamente!")


_validar_construir_prompt_rag()

✓ incluye el texto de cada fragmento
✓ numera desde 1 y con la fuente: '[1] (fuente: a.txt)', '[2] (fuente: b.txt)'
✓ termina con 'Pregunta: <pregunta>'
✓ sin fragmentos, el prompt es solo la pregunta
✓ formato exacto (bloques separados por una linea en blanco)

🎉 ¡Todo funciona correctamente!


Este es el system prompt del asistente. Tiene tres reglas: responder **solo** con los fragmentos, **citar** el numero de cada uno y admitir cuando la respuesta **no esta**, con una frase fija que despues podemos detectar. Ejecuta la celda para ver un prompt real.

In [19]:
SYSTEM_RAG = (
    "Eres el asistente interno de Transportes del Llano. "
    "Responde en espanol, de forma breve, y SOLO con base en los fragmentos que se te entregan. "
    "Despues de cada dato, cita entre corchetes el numero del fragmento de donde lo sacaste, por ejemplo [2]. "
    "Si la respuesta no esta en los fragmentos, responde exactamente: "
    "'No encuentro esa informacion en los documentos.'"
)

p = "¿Cuanto se paga de viaticos en ruta nacional?"
print(construir_prompt_rag(p, buscador_semantico(p, k=2)))

[1] (fuente: politica_viaticos.txt)
TRANSPORTES DEL LLANO S.A.S. POLÍTICA DE VIÁTICOS Y GASTOS DE VIAJE Código: VG-004 — Vigente desde enero de 2026 1. DEFINICIÓN Los viáticos son el dinero que la empresa entrega al conductor para cubrir su alimentación durante los viajes. No hacen parte del salario y no se entregan en los días sin viaje. 2. VALOR DE LOS VIÁTICOS En ruta regional, es decir, en viajes dentro del Meta y el Casanare, el viático es de 45.000 pesos por día de viaje. En ruta nacional, es decir, en viajes que salen del Meta y el Casanare, por ejemplo hacia Bogotá, Tunja o la Costa, el viático es de 85.000 pesos por día de viaje. Un día de viaje se cuenta cuando

[2] (fuente: politica_viaticos.txt)
tiene camarote y está en un parqueadero vigilado; en ese caso recibe un auxilio de 40.000 pesos por noche. 4. PEAJES Y COMBUSTIBLE Los peajes se pagan con el tag electrónico instalado en cada vehículo. La empresa no reembolsa peajes pagados en efectivo, salvo que el tag haya fallado

#### Tarea 2: Construir la funcion `responder_con_rag`

Esta funcion recibe el **buscador como parametro**: cualquier funcion `(pregunta, k) -> fragmentos`, como `buscador_semantico` o `buscador_bm25`. Asi podemos cambiar de buscador sin tocar esta funcion (en la Actividad 5 lo cambiaremos por uno de ChromaDB).

### Requisitos

1. Recuperar `k` fragmentos con `buscador(pregunta, k)`.
2. Armar el prompt con `construir_prompt_rag`.
3. Llamar a `preguntar_groq` con `system_prompt=SYSTEM_RAG` y `temperature=0` (queremos datos, no creatividad).
4. Devolver un diccionario `{"respuesta": <texto del LLM>, "fragmentos": <lo recuperado>}`. Devolvemos los fragmentos para poder mostrar las fuentes.

In [20]:
def responder_con_rag(pregunta: str, buscador, k: int = 3) -> dict:
  """Recupera k fragmentos con `buscador`, arma el prompt y le pide la respuesta al LLM."""

  # TODO: recuperar los fragmentos con el buscador
  recuperados = buscador(pregunta, k)

  # TODO: armar el prompt y llamar a preguntar_groq con SYSTEM_RAG y temperature=0
  prompt = construir_prompt_rag(pregunta, recuperados)
  respuesta = preguntar_groq(prompt, system_prompt=SYSTEM_RAG, temperature=0)

  # TODO: devolver la respuesta y los fragmentos usados
  return {"respuesta": respuesta, "fragmentos": recuperados}

**Evaluacion de implementacion**

Esta validacion simula el buscador y el LLM, asi que no gasta tokens.

In [21]:
# @title
# Celda de validacion. No modificar.
from unittest.mock import patch

def _validar_responder_con_rag():
  for nombre in ("construir_prompt_rag", "responder_con_rag", "SYSTEM_RAG"):
    if nombre not in globals():
      print(f"✗ Todavia no existe '{nombre}'. Ejecuta las celdas anteriores.")
      return

  frags = [
      {"id": "x.txt#0", "fuente": "x.txt", "texto": "La clave secreta es 4242.", "score": 0.9},
      {"id": "y.txt#1", "fuente": "y.txt", "texto": "El cafe se sirve a las 10.", "score": 0.8},
  ]
  llamadas_buscador = []

  def _buscador_falso(pregunta, k=3):
    llamadas_buscador.append((pregunta, k))
    return frags[:k]

  fallas = []
  with patch("__main__.preguntar_groq", return_value="Es 4242 [1].") as mock_groq:
    try:
      obtenido = responder_con_rag("¿Cual es la clave?", _buscador_falso, k=2)
    except Exception as e:
      print(f"✗ responder_con_rag(...) lanzo {type(e).__name__}: {e}")
      return

    if llamadas_buscador == [("¿Cual es la clave?", 2)]:
      print("✓ Llama al buscador una vez, con la pregunta y k")
    else:
      print(f"✗ Se esperaba buscador('¿Cual es la clave?', 2), se llamo con: {llamadas_buscador}")
      fallas.append("buscador")

    if not mock_groq.called:
      print("✗ No se llamo a 'preguntar_groq'")
      fallas.append("groq")
    else:
      args, kwargs = mock_groq.call_args
      prompt_usado = args[0] if args else kwargs.get("prompt", "")
      if "La clave secreta es 4242." in prompt_usado and "El cafe se sirve a las 10." in prompt_usado and "¿Cual es la clave?" in prompt_usado:
        print("✓ El prompt incluye los fragmentos recuperados y la pregunta")
      else:
        print("✗ El prompt debe incluir los fragmentos y la pregunta. ¿Usaste construir_prompt_rag?")
        fallas.append("prompt")

      if kwargs.get("system_prompt") == SYSTEM_RAG:
        print("✓ Usa system_prompt=SYSTEM_RAG")
      else:
        print(f"✗ Falta system_prompt=SYSTEM_RAG (se paso {kwargs.get('system_prompt')!r})")
        fallas.append("system")

      if kwargs.get("temperature") == 0:
        print("✓ Usa temperature=0")
      else:
        print(f"✗ Falta temperature=0 (se paso {kwargs.get('temperature')!r})")
        fallas.append("temperature")

  if not isinstance(obtenido, dict):
    print(f"✗ Debia devolver un dict {{'respuesta': ..., 'fragmentos': ...}}, devolvio {obtenido!r}")
    fallas.append("retorno")
  else:
    if obtenido.get("respuesta") == "Es 4242 [1].":
      print("✓ 'respuesta' es el texto que devolvio el LLM")
    else:
      print(f"✗ 'respuesta' debia ser el texto del LLM, es {obtenido.get('respuesta')!r}")
      fallas.append("respuesta")
    if obtenido.get("fragmentos") == frags[:2]:
      print("✓ 'fragmentos' son los que devolvio el buscador")
    else:
      print(f"✗ 'fragmentos' debian ser los que devolvio el buscador, son {obtenido.get('fragmentos')!r}")
      fallas.append("fragmentos")

  print()
  if fallas:
    print(f"{len(fallas)} problema(s) por corregir.")
  else:
    print("\U0001F389 ¡Todo funciona correctamente!")


_validar_responder_con_rag()

✓ Llama al buscador una vez, con la pregunta y k
✓ El prompt incluye los fragmentos recuperados y la pregunta
✓ Usa system_prompt=SYSTEM_RAG
✓ Usa temperature=0
✓ 'respuesta' es el texto que devolvio el LLM
✓ 'fragmentos' son los que devolvio el buscador

🎉 ¡Todo funciona correctamente!


#### Tarea 3: Las preguntas del Workshop 3, ahora con RAG

Esta celda si usa tu API key. Las tres primeras preguntas son las del Workshop 3; la cuarta es la que alla respondia "Eso no esta en el manual".

In [22]:
def mostrar_respuesta(pregunta: str, resultado: dict):
  print(f"P: {pregunta}")
  print(f"R: {resultado['respuesta']}")
  usados = ", ".join(f"[{i}] {f['id']}" for i, f in enumerate(resultado["fragmentos"], start=1))
  print(f"   fragmentos: {usados}\n")


preguntas_w3 = [
    "¿Cual es la carga maxima de fruto de palma por camion en Transportes del Llano?",
    "¿Cada cuantos kilometros se hace el mantenimiento preventivo?",
    "¿Cuanto se paga de viaticos en ruta nacional?",
    "¿Cuantos dias de vacaciones tienen los conductores?",
]

for pregunta in preguntas_w3:
  mostrar_respuesta(pregunta, responder_con_rag(pregunta, buscador_semantico))

P: ¿Cual es la carga maxima de fruto de palma por camion en Transportes del Llano?
R: La carga máxima de fruto de palma por camión sencillo es de 12 toneladas. [1]
   fragmentos: [1] manual_operativo.txt#2, [2] tarifas_servicio.txt#0, [3] manual_operativo.txt#0

P: ¿Cada cuantos kilometros se hace el mantenimiento preventivo?
R: Se realiza cada 10.000 kilómetros recorridos. [1]
   fragmentos: [1] politica_mantenimiento.txt#1, [2] politica_mantenimiento.txt#2, [3] politica_mantenimiento.txt#0

P: ¿Cuanto se paga de viaticos en ruta nacional?
R: Se paga 85.000 pesos por día de viaje en ruta nacional. [1]
   fragmentos: [1] politica_viaticos.txt#0, [2] politica_viaticos.txt#2, [3] politica_viaticos.txt#1

P: ¿Cuantos dias de vacaciones tienen los conductores?
R: Los conductores tienen derecho a 15 días hábiles de vacaciones remuneradas por cada año de servicio. [1]
   fragmentos: [1] reglamento_interno.txt#1, [2] politica_viaticos.txt#1, [3] politica_viaticos.txt#0



Las tres primeras deberian salir igual que en el Workshop 3 (12 toneladas, 10.000 km, 85.000 pesos), ahora con cita. La de vacaciones deberia responder **15 dias habiles**: no porque el modelo sea mejor, sino porque ahora tiene mas documentos y el buscador encontro el correcto.

Ahora las peligrosas: preguntas cuya respuesta **no esta**, pero que se parecen mucho a cosas que si estan.

In [23]:
trampas = [
    "¿Cual es el salario de un conductor?",                 # la palabra 'salario' si aparece, el dato no
    "¿Cuanto cuesta un flete de Villavicencio a Medellin?",   # hay tarifas, pero no para esa ruta
]

for pregunta in trampas:
  mostrar_respuesta(pregunta, responder_con_rag(pregunta, buscador_semantico))

P: ¿Cual es el salario de un conductor?
R: No encuentro esa información en los documentos.
   fragmentos: [1] reglamento_interno.txt#0, [2] politica_viaticos.txt#0, [3] reglamento_interno.txt#1

P: ¿Cuanto cuesta un flete de Villavicencio a Medellin?
R: No encuentro esa informacion en los documentos.
   fragmentos: [1] tarifas_servicio.txt#0, [2] tarifas_servicio.txt#2, [3] politica_viaticos.txt#1



Fijate en algo importante: **el buscador siempre devuelve `k` fragmentos**, aunque ninguno sirva. No sabe decir "no hay nada". Todo depende de que el LLM respete la instruccion de admitirlo. Revisa si lo hizo, o si "estimo" una tarifa a Medellin a partir de las otras rutas: eso seria una alucinacion con cita, la mas dificil de detectar.

#### Tarea 4: ¿Cuanto texto nos ahorramos?

Esta celda esta completa. Compara lo que le mandariamos al LLM con el metodo del Workshop 3 (todo el corpus en cada pregunta) contra el RAG.

In [24]:
palabras_corpus = sum(len(d["texto"].split()) for d in documentos)
palabras_rag = [len(construir_prompt_rag(p, buscador_semantico(p, k=3)).split()) for p in preguntas_w3]
promedio_rag = sum(palabras_rag) / len(palabras_rag)
palabras_por_doc = palabras_corpus / len(documentos)

print(f"Workshop 3 (todo el corpus en el prompt): {palabras_corpus:>9,.0f} palabras por pregunta")
print(f"RAG con k=3:                              {promedio_rag:>9,.0f} palabras por pregunta")
print(f"\nCon {len(documentos)} documentos, el RAG manda {palabras_corpus / promedio_rag:.0f} veces menos texto.")
print(f"Con 500 documentos como estos (~{500 * palabras_por_doc:,.0f} palabras) seguiria mandando ~{promedio_rag:,.0f}: "
      f"unas {500 * palabras_por_doc / promedio_rag:,.0f} veces menos.")

Workshop 3 (todo el corpus en el prompt):     3,115 palabras por pregunta
RAG con k=3:                                    380 palabras por pregunta

Con 7 documentos, el RAG manda 8 veces menos texto.
Con 500 documentos como estos (~222,500 palabras) seguiria mandando ~380: unas 586 veces menos.


## Actividad 4: Evaluar la recuperacion

### Objetivo

Hasta ahora juzgamos "a ojo" con tres o cuatro preguntas. Asi no hay forma de saber si un cambio (otro tamano de fragmento, otro buscador, otro modelo de embeddings) **mejora o empeora** el sistema.

En un RAG pueden fallar dos cosas:

1. **El buscador** no trae el fragmento correcto.
2. **El LLM** no usa bien lo que se le entrega.

La primera se puede medir sin gastar ni un token del LLM, y es la que mas pesa: si el fragmento correcto no llega al prompt, el LLM no tiene como responder bien (o peor: inventa).

### El conjunto de evaluacion

Es una lista de preguntas con su **evidencia**: un pedazo de texto corto que solo aparece en el fragmento que responde la pregunta. Un fragmento recuperado es "correcto" si contiene la evidencia. Varias preguntas estan escritas **a proposito sin las palabras del documento**, como las haria un conductor.

### Las metricas

- **Hit rate@k**: fraccion de preguntas en las que **al menos uno** de los `k` fragmentos recuperados es correcto.
- **MRR** (*Mean Reciprocal Rank*): para cada pregunta se toma `1 / posicion` del **primer** fragmento correcto (1 si sale primero, 0.5 si sale segundo, 0.33 si sale tercero, 0 si no aparece), y se promedia. Premia que el correcto salga **arriba**.

| Pregunta | Posicion del primer correcto | 1 / posicion |
|---|---|---|
| A | 1 | 1 |
| B | 3 | 0.33 |
| C | no aparece en los k | 0 |

Con k=3: hit rate = 2/3 = 0.67 y MRR = (1 + 0.33 + 0) / 3 = 0.44.

In [25]:
CASOS_EVAL = [
    {"pregunta": "¿Cuanto es lo maximo de fruto de palma que puede llevar un camion sencillo?", "evidencia": "12 toneladas"},
    {"pregunta": "¿Cada cuanto se le hace mantenimiento preventivo a los vehiculos?", "evidencia": "10.000 kilómetros"},
    {"pregunta": "¿Cuantos dias de vacaciones tienen los conductores?", "evidencia": "15 días hábiles"},
    {"pregunta": "¿A que velocidad maxima puede ir un camion en carretera?", "evidencia": "80 km/h"},
    {"pregunta": "¿A que numero llamo si tengo una emergencia en la via?", "evidencia": "601 555 0147"},
    {"pregunta": "¿Se puede manejar de noche por la via al Llano?", "evidencia": "10:00 p. m."},
    {"pregunta": "¿Que licencia necesita un conductor para trabajar en la empresa?", "evidencia": "C3"},
    {"pregunta": "¿Que hago si se riega aceite en la carretera?", "evidencia": "kit antiderrames"},
    {"pregunta": "¿Cuanto tiempo puede pasar entre la cosecha del fruto y la entrega en planta?", "evidencia": "superar las 24 horas"},
    # Escritas sin las palabras del documento (como preguntaria un conductor):
    {"pregunta": "¿Cuanta plata le dan a un conductor por dia cuando viaja fuera de la region?", "evidencia": "85.000"},
    {"pregunta": "¿Cuanto le cobramos al cliente por cada hora extra que el camion se queda esperando?", "evidencia": "90.000"},
    {"pregunta": "¿Que tan gastadas pueden estar las llantas antes de cambiarlas?", "evidencia": "3 milímetros"},
    {"pregunta": "¿Cuantas veces al año entregan la ropa de trabajo?", "evidencia": "tres veces al año"},
    {"pregunta": "¿Cuanto tiempo tengo para entregar las facturas de los gastos del viaje?", "evidencia": "cinco días hábiles"},
    {"pregunta": "¿Me puedo tomar una cerveza antes de salir a ruta?", "evidencia": "alcohol en la sangre"},
    {"pregunta": "¿Que hago si me roban el camion?", "evidencia": "No oponga resistencia"},
    {"pregunta": "¿Que tan rapido puedo ir dentro de una finca?", "evidencia": "10 km/h"},
]

print(f"{len(CASOS_EVAL)} casos de evaluacion")

17 casos de evaluacion


#### Tarea 1: Construir la funcion `evaluar_recuperacion`

### Requisitos

1. Para cada caso, pedirle al buscador `buscador(caso["pregunta"], k)`.
2. Un fragmento es correcto si contiene la evidencia, **sin importar mayusculas**: `caso["evidencia"].lower() in fragmento["texto"].lower()`.
3. Para el MRR cuenta solo el **primer** fragmento correcto de cada pregunta.
4. Devolver `{"hit_rate": ..., "mrr": ..., "fallos": [...]}`, donde `"fallos"` es la lista de preguntas en las que **ningun** fragmento fue correcto.

In [27]:
def evaluar_recuperacion(buscador, casos: list, k: int = 3) -> dict:
  """Calcula hit rate@k y MRR de un buscador sobre un conjunto de casos con evidencia."""

  aciertos = 0
  suma_rr = 0.0
  fallos = []

  # TODO: para cada caso, buscar la posicion (desde 1) del primer fragmento que contiene la evidencia
  #       y acumular aciertos, suma de 1/posicion y fallos
  for caso in casos:
    resultados = buscador(caso["pregunta"], k)
    evidencia = caso["evidencia"].lower()

    posicion = None
    for i, fragmento in enumerate(resultados, start=1):
      if evidencia in fragmento["texto"].lower():
        posicion = i
        break

    if posicion is None:
      fallos.append(caso["pregunta"])
    else:
      aciertos += 1
      suma_rr += 1 / posicion

  # TODO: devolver el diccionario con hit_rate, mrr y fallos
  return {"hit_rate": aciertos / len(casos), "mrr": suma_rr / len(casos), "fallos": fallos}

**Evaluacion de implementacion**

In [28]:
# @title
# Celda de validacion. No modificar.
from math import isclose

def _validar_evaluar_recuperacion():
  if "evaluar_recuperacion" not in globals():
    print("✗ Todavia no existe 'evaluar_recuperacion'. Ejecuta la celda anterior.")
    return

  textos_por_pregunta = {
      "p1": ["aqui esta la CLAVE", "nada", "nada"],          # correcto en la posicion 1 (y en mayusculas)
      "p2": ["nada", "tampoco", "por fin la clave"],          # posicion 3
      "p3": ["nada", "nada", "nada"],                         # no aparece
      "p4": ["nada", "la clave", "otra vez la clave"],        # posicion 2 (solo cuenta el primero)
  }
  ks_recibidos = []

  def _buscador_falso(pregunta, k=3):
    ks_recibidos.append(k)
    return [{"id": f"x#{i}", "fuente": "x.txt", "texto": t} for i, t in enumerate(textos_por_pregunta[pregunta][:k])]

  casos = [{"pregunta": p, "evidencia": "clave"} for p in textos_por_pregunta]
  fallas = []

  try:
    r3 = evaluar_recuperacion(_buscador_falso, casos, k=3)
    r1 = evaluar_recuperacion(_buscador_falso, casos, k=1)
    r_may = evaluar_recuperacion(_buscador_falso, [{"pregunta": "p2", "evidencia": "CLAVE"}], k=3)
  except Exception as e:
    print(f"✗ evaluar_recuperacion(...) lanzo {type(e).__name__}: {e}")
    return

  if not isinstance(r3, dict) or not {"hit_rate", "mrr", "fallos"} <= set(r3):
    print(f"✗ Debia devolver un dict con 'hit_rate', 'mrr' y 'fallos', devolvio {r3!r}")
    return

  def revisar(descripcion, obtenido, esperado):
    ok = isclose(obtenido, esperado, abs_tol=1e-6) if isinstance(esperado, float) else obtenido == esperado
    if ok:
      print(f"✓ {descripcion}")
    else:
      print(f"✗ {descripcion}\n    esperaba: {esperado}\n    obtuvo:   {obtenido}")
      fallas.append(descripcion)

  revisar("le pasa k al buscador", 3 in ks_recibidos and 1 in ks_recibidos, True)
  revisar("k=3: hit rate = 3/4 (p3 no tiene la evidencia)", r3["hit_rate"], 0.75)
  revisar("k=3: MRR = (1 + 1/3 + 0 + 1/2) / 4  (en p4 solo cuenta el primer correcto)", r3["mrr"], (1 + 1/3 + 0 + 1/2) / 4)
  revisar("k=3: fallos = ['p3']", r3["fallos"], ["p3"])
  revisar("k=1: hit rate = 1/4", r1["hit_rate"], 0.25)
  revisar("k=1: MRR = 1/4", r1["mrr"], 0.25)
  revisar("k=1: fallos = ['p2', 'p3', 'p4']", r1["fallos"], ["p2", "p3", "p4"])
  revisar("no importan las mayusculas de la evidencia", r_may["hit_rate"], 1.0)

  print()
  if fallas:
    print(f"{len(fallas)} problema(s) por corregir.")
  else:
    print("\U0001F389 ¡Todo funciona correctamente!")


_validar_evaluar_recuperacion()

✓ le pasa k al buscador
✓ k=3: hit rate = 3/4 (p3 no tiene la evidencia)
✓ k=3: MRR = (1 + 1/3 + 0 + 1/2) / 4  (en p4 solo cuenta el primer correcto)
✓ k=3: fallos = ['p3']
✓ k=1: hit rate = 1/4
✓ k=1: MRR = 1/4
✓ k=1: fallos = ['p2', 'p3', 'p4']
✓ no importan las mayusculas de la evidencia

🎉 ¡Todo funciona correctamente!


#### Tarea 2: BM25 contra semantico

Esta celda esta completa. Evalua los dos buscadores con distintos `k`.

In [29]:
import pandas as pd

filas = []
for nombre, buscador in [("BM25", buscador_bm25), ("Semantico", buscador_semantico)]:
  for k in [1, 3, 5]:
    r = evaluar_recuperacion(buscador, CASOS_EVAL, k=k)
    filas.append({"buscador": nombre, "k": k, "hit_rate": r["hit_rate"], "mrr": r["mrr"]})

tabla = pd.DataFrame(filas).pivot(index="k", columns="buscador", values=["hit_rate", "mrr"]).round(2)
display(tabla)

for nombre, buscador in [("BM25", buscador_bm25), ("Semantico", buscador_semantico)]:
  print(f"\nFallos de {nombre} con k=3:")
  for pregunta in evaluar_recuperacion(buscador, CASOS_EVAL, k=3)["fallos"] or ["(ninguno)"]:
    print(f"  - {pregunta}")

hit_rate             mrr          
buscador     BM25 Semantico  BM25 Semantico
k                                          
1            0.59      0.88  0.59      0.88
3            0.82      0.94  0.71      0.91
5            0.82      0.94  0.71      0.91


Fallos de BM25 con k=3:
  - ¿Se puede manejar de noche por la via al Llano?
  - ¿Me puedo tomar una cerveza antes de salir a ruta?
  - ¿Que hago si me roban el camion?

Fallos de Semantico con k=3:
  - ¿Se puede manejar de noche por la via al Llano?


Mira **en que preguntas** falla cada uno, no solo el promedio. BM25 suele fallar en las que no comparten palabras con el documento; el semantico, en las que dependen de un dato exacto. Si fallan en preguntas distintas, combinarlos (**busqueda hibrida**) deberia ganarle a los dos: es una de las extensiones.

Y un recordatorio honesto: son 17 preguntas. Una diferencia de 0.06 en el hit rate es **una sola pregunta**. Para decidir en serio se necesitan cientos de casos.

#### Tarea 3: ¿Que tamano de fragmento conviene?

Esta celda esta completa. Reconstruye el indice semantico con fragmentos de 50, 120 y 300 palabras y los evalua con el mismo conjunto. La ultima columna es una aproximacion de cuanto texto le llegaria al LLM con k=3.

In [30]:
filas = []
for tamano in [50, 120, 300]:
  frags_exp = fragmentar_corpus(documentos, tamano=tamano, solapamiento=tamano // 4)
  embs_exp = embeber([f["texto"] for f in frags_exp], tipo="passage")

  def buscador_exp(pregunta, k=3):
    return buscar(pregunta, frags_exp, embs_exp, k)

  r1 = evaluar_recuperacion(buscador_exp, CASOS_EVAL, k=1)
  r3 = evaluar_recuperacion(buscador_exp, CASOS_EVAL, k=3)
  filas.append({
      "tamano": tamano,
      "n_fragmentos": len(frags_exp),
      "hit_rate@1": round(r1["hit_rate"], 2),
      "hit_rate@3": round(r3["hit_rate"], 2),
      "mrr@3": round(r3["mrr"], 2),
      "palabras al LLM (k=3)": sum(len(f["texto"].split()) for f in frags_exp[:3]),
  })

display(pd.DataFrame(filas))

,tamano,n_fragmentos,hit_rate@1,hit_rate@3,mrr@3,palabras al LLM (k=3)
0,50,84,0.82,0.94,0.88,150
1,120,35,0.88,0.94,0.91,360
2,300,15,0.65,0.94,0.79,876


Con fragmentos grandes el hit rate suele subir... pero es un poco **enganoso**. Con 300 palabras cada documento queda en 2 o 3 fragmentos, asi que "acertar" es mucho mas facil, y a cambio le mandas al LLM cerca de 900 palabras donde el dato ocupa una linea. Ademas, 300 palabras en espanol ya estan cerca del limite de 512 tokens de e5: con fragmentos mas grandes, el final se **cortaria sin avisar** y nunca se podria encontrar.

Con fragmentos chicos el buscador es mas preciso, pero una frase puede quedar sin el contexto que la explica.

No existe un tamano correcto universal: **se escoge midiendo**, con un conjunto de evaluacion como este y mirando tambien el costo.

## Actividad 5: Base de datos vectorial

### Objetivo

Nuestro indice es una matriz de numpy en memoria. Funciona con unas decenas de fragmentos, pero en produccion tiene cuatro problemas:

1. **Se pierde al reiniciar**: hay que volver a embeber todo.
2. **Compara la pregunta contra todos los fragmentos**: con millones, es lento.
3. **No permite filtrar**: por ejemplo, "busca solo en el protocolo de emergencias".
4. **Agregar un documento** implica reconstruir la matriz.

Una **base de datos vectorial** resuelve eso. Usamos **ChromaDB** porque corre dentro del notebook sin instalar servidores; en produccion hay alternativas como pgvector (sobre Postgres), Qdrant, Weaviate o Pinecone. Los conceptos son los mismos:

- **Coleccion**: como una tabla. Cada registro tiene `id`, documento (el texto), embedding y metadatos.
- **Metadatos**: datos extra de cada registro (aqui, la fuente) que sirven para **filtrar**.
- **Distancia**: Chroma devuelve distancias, no similitudes. Con distancia coseno, `distancia = 1 - similitud`.
- **Indice aproximado (HNSW)**: en vez de comparar contra todo, recorre un grafo de vecinos y encuentra los mas cercanos casi siempre, muchisimo mas rapido.

Ejecuta la celda para crear la coleccion.

In [31]:
import chromadb

cliente_chroma = chromadb.PersistentClient(path="indice_rag")   # se guarda en disco, en la carpeta indice_rag/
NOMBRE_COLECCION = "transportes_del_llano"

# Borramos la coleccion si ya existia, para que puedas correr esta celda varias veces sin mezclar nada
try:
  cliente_chroma.delete_collection(NOMBRE_COLECCION)
except Exception:
  pass

coleccion = cliente_chroma.create_collection(
    name=NOMBRE_COLECCION,
    configuration={"hnsw": {"space": "cosine"}},   # medir con distancia coseno
)
print(f"✓ Coleccion '{coleccion.name}' creada con {coleccion.count()} registros")

✓ Coleccion 'transportes_del_llano' creada con 0 registros


#### Tarea 1: Construir la funcion `indexar_en_chroma`

### Requisitos

1. Guardar todos los fragmentos con `coleccion.upsert(...)`, pasando cuatro listas del mismo largo:
   - `ids`: el `id` de cada fragmento.
   - `documents`: el `texto` de cada fragmento.
   - `embeddings`: un vector por fragmento. Chroma prefiere listas: usa `embeddings.tolist()`.
   - `metadatas`: un diccionario por fragmento, `{"fuente": ...}`.
2. Usa `upsert` y no `add`. *Upsert* = *update* + *insert*: si el `id` ya existe lo reemplaza, y si no, lo crea. Asi, reindexar un documento corregido actualiza sus fragmentos en vez de ignorar el cambio.

In [32]:
def indexar_en_chroma(coleccion, fragmentos: list, embeddings: np.ndarray):
  """Guarda los fragmentos (con su embedding y su fuente) en una coleccion de Chroma."""

  # TODO: llamar a coleccion.upsert con ids, documents, embeddings y metadatas
  coleccion.upsert(
      ids=[f["id"] for f in fragmentos],
      documents=[f["texto"] for f in fragmentos],
      embeddings=np.asarray(embeddings).tolist(),
      metadatas=[{"fuente": f["fuente"]} for f in fragmentos],
  )

**Evaluacion de implementacion**

Usa una coleccion temporal aparte, asi que no toca la del corpus.

In [33]:
# @title
# Celda de validacion. No modificar.
import uuid

def _validar_indexar_en_chroma():
  if "indexar_en_chroma" not in globals():
    print("✗ Todavia no existe 'indexar_en_chroma'. Ejecuta la celda anterior.")
    return

  prueba = chromadb.EphemeralClient().create_collection(
      f"prueba_{uuid.uuid4().hex[:8]}", configuration={"hnsw": {"space": "cosine"}})
  frags = [
      {"id": "a.txt#0", "fuente": "a.txt", "texto": "texto uno"},
      {"id": "a.txt#1", "fuente": "a.txt", "texto": "texto dos"},
      {"id": "b.txt#0", "fuente": "b.txt", "texto": "texto tres"},
  ]
  embs = np.eye(3)

  fallas = []
  try:
    indexar_en_chroma(prueba, frags, embs)
  except Exception as e:
    print(f"✗ indexar_en_chroma(...) lanzo {type(e).__name__}: {e}")
    return

  if prueba.count() == 3:
    print("✓ Guarda los 3 fragmentos")
  else:
    print(f"✗ La coleccion debia tener 3 registros, tiene {prueba.count()}")
    print(f"\n{len(fallas) + 1} problema(s) por corregir.")
    return

  guardado = prueba.get(ids=["b.txt#0"], include=["documents", "metadatas", "embeddings"])
  if guardado["documents"] == ["texto tres"]:
    print("✓ Guarda el texto de cada fragmento en 'documents'")
  else:
    print(f"✗ El documento de 'b.txt#0' debia ser 'texto tres', es {guardado['documents']}")
    fallas.append("documents")

  if guardado["metadatas"] and guardado["metadatas"][0] and guardado["metadatas"][0].get("fuente") == "b.txt":
    print("✓ Guarda la fuente en los metadatos")
  else:
    print(f"✗ Los metadatos de 'b.txt#0' debian ser {{'fuente': 'b.txt'}}, son {guardado['metadatas']}")
    fallas.append("metadatas")

  if np.allclose(np.asarray(guardado["embeddings"][0]), [0, 0, 1]):
    print("✓ Guarda el embedding que corresponde a cada fragmento")
  else:
    print(f"✗ El embedding de 'b.txt#0' debia ser [0, 0, 1], es {guardado['embeddings'][0]}")
    fallas.append("embeddings")

  frags_corregidos = [dict(frags[0], texto="texto uno CORREGIDO")]
  try:
    indexar_en_chroma(prueba, frags_corregidos, np.eye(3)[:1])
  except Exception as e:
    print(f"✗ Reindexar un fragmento lanzo {type(e).__name__}: {e}")
    fallas.append("reindexar")
  else:
    doc = prueba.get(ids=["a.txt#0"])["documents"][0]
    if prueba.count() == 3 and doc == "texto uno CORREGIDO":
      print("✓ Reindexar un fragmento lo actualiza sin duplicarlo (upsert)")
    else:
      print(f"✗ Al reindexar 'a.txt#0' el texto debia cambiar a 'texto uno CORREGIDO', quedo {doc!r}. ¿Usaste add en vez de upsert?")
      fallas.append("upsert")

  print()
  if fallas:
    print(f"{len(fallas)} problema(s) por corregir.")
  else:
    print("\U0001F389 ¡Todo funciona correctamente!")


_validar_indexar_en_chroma()

✓ Guarda los 3 fragmentos
✓ Guarda el texto de cada fragmento en 'documents'
✓ Guarda la fuente en los metadatos
✓ Guarda el embedding que corresponde a cada fragmento
✓ Reindexar un fragmento lo actualiza sin duplicarlo (upsert)

🎉 ¡Todo funciona correctamente!


#### Tarea 2: Construir la funcion `buscar_chroma`

### Requisitos

1. Embeber la pregunta con `tipo="query"`, igual que en `buscar`.
2. Consultar con `coleccion.query(query_embeddings=[...], n_results=k, where=filtro)`, donde `filtro` es `{"fuente": fuente}` si se pidio una fuente, o `None` si no.
3. Chroma responde con **listas de listas** (una lista por cada pregunta enviada; como mandamos una sola, toma el `[0]`):
   ```
   r["ids"][0]        -> ["politica_viaticos.txt#1", ...]
   r["documents"][0]  -> ["En ruta regional, ...", ...]
   r["metadatas"][0]  -> [{"fuente": "politica_viaticos.txt"}, ...]
   r["distances"][0]  -> [0.12, 0.18, ...]
   ```
4. Devolver el **mismo formato** que `buscar`: una lista de diccionarios con `id`, `fuente`, `texto` y `score`, donde `score = 1 - distancia`. Asi, `buscar_chroma` puede reemplazar a `buscar` en todo lo que ya construimos.

In [36]:
def buscar_chroma(pregunta: str, coleccion, k: int = 3, fuente: str = None) -> list:
  """Busca los k fragmentos mas parecidos en Chroma, opcionalmente solo dentro de una fuente."""

  # TODO: embeber la pregunta y armar el filtro por fuente (o None)
  vector_pregunta = embeber([pregunta], tipo="query")[0]
  filtro = {"fuente": fuente} if fuente else None

  # TODO: consultar la coleccion
  r = coleccion.query(query_embeddings=[np.asarray(vector_pregunta).tolist()], n_results=k, where=filtro)

  # TODO: convertir la respuesta al formato de buscar (id, fuente, texto, score = 1 - distancia)
  return [
      {"id": id_, "fuente": meta["fuente"], "texto": texto, "score": 1 - distancia}
      for id_, texto, meta, distancia in zip(r["ids"][0], r["documents"][0], r["metadatas"][0], r["distances"][0])
  ]

**Evaluacion de implementacion**

In [37]:
# @title
# Celda de validacion. No modificar.
import uuid
from unittest.mock import patch

def _validar_buscar_chroma():
  if "buscar_chroma" not in globals():
    print("✗ Todavia no existe 'buscar_chroma'. Ejecuta la celda anterior.")
    return

  prueba = chromadb.EphemeralClient().create_collection(
      f"prueba_{uuid.uuid4().hex[:8]}", configuration={"hnsw": {"space": "cosine"}})
  prueba.add(
      ids=["a.txt#0", "a.txt#1", "b.txt#0", "b.txt#1"],
      documents=["identico", "parecido", "algo parecido", "nada que ver"],
      embeddings=[[1, 0, 0], [0.8, 0.6, 0], [0.6, 0.8, 0], [0, 0, 1]],
      metadatas=[{"fuente": "a.txt"}, {"fuente": "a.txt"}, {"fuente": "b.txt"}, {"fuente": "b.txt"}],
  )
  # similitudes con [1, 0, 0]: 1.0, 0.8, 0.6, 0.0

  llamadas = []

  def _embeber_falso(textos, tipo="passage"):
    llamadas.append(tipo)
    return np.array([[1.0, 0.0, 0.0]])

  fallas = []
  with patch("__main__.embeber", _embeber_falso):
    try:
      r2 = buscar_chroma("pregunta", prueba, k=2)
      rb = buscar_chroma("pregunta", prueba, k=3, fuente="b.txt")
    except Exception as e:
      print(f"✗ buscar_chroma(...) lanzo {type(e).__name__}: {e}")
      return

  if llamadas and llamadas[0] == "query":
    print("✓ Embebe la pregunta con tipo='query'")
  else:
    print(f"✗ La pregunta debe embeberse con tipo='query' (llamadas: {llamadas})")
    fallas.append("tipo")

  if not isinstance(r2, list) or not r2 or not all(isinstance(x, dict) for x in r2):
    print(f"✗ Debia devolver una lista de diccionarios, devolvio {r2!r}")
    print(f"\n{len(fallas) + 1} problema(s) por corregir.")
    return

  if [x.get("id") for x in r2] == ["a.txt#0", "a.txt#1"]:
    print("✓ Devuelve los k=2 mas parecidos, en orden")
  else:
    print(f"✗ Esperaba ['a.txt#0', 'a.txt#1'], obtuvo {[x.get('id') for x in r2]}")
    fallas.append("orden")

  if all(set(x) >= {"id", "fuente", "texto", "score"} for x in r2):
    print("✓ Cada resultado tiene id, fuente, texto y score")
  else:
    print(f"✗ Cada resultado debe tener id, fuente, texto y score. Llaves: {[sorted(x) for x in r2]}")
    fallas.append("llaves")

  if r2[0].get("fuente") == "a.txt" and r2[0].get("texto") == "identico":
    print("✓ 'fuente' sale de los metadatos y 'texto' de los documentos")
  else:
    print(f"✗ El primer resultado debia tener fuente 'a.txt' y texto 'identico': {r2[0]}")
    fallas.append("campos")

  scores = [x.get("score") for x in r2]
  if all(isinstance(s, (int, float)) for s in scores) and np.allclose(scores, [1.0, 0.8], atol=1e-3):
    print(f"✓ score = 1 - distancia: {np.round(scores, 3).tolist()}")
  else:
    print(f"✗ Los score debian ser ~[1.0, 0.8] (1 - distancia), son {scores}")
    fallas.append("score")

  if [x.get("id") for x in rb] == ["b.txt#0", "b.txt#1"]:
    print("✓ Con fuente='b.txt' solo devuelve fragmentos de esa fuente")
  else:
    print(f"✗ Con fuente='b.txt' esperaba ['b.txt#0', 'b.txt#1'], obtuvo {[x.get('id') for x in rb]}")
    fallas.append("filtro")

  print()
  if fallas:
    print(f"{len(fallas)} problema(s) por corregir.")
  else:
    print("\U0001F389 ¡Todo funciona correctamente!")


_validar_buscar_chroma()

✓ Embebe la pregunta con tipo='query'
✓ Devuelve los k=2 mas parecidos, en orden
✓ Cada resultado tiene id, fuente, texto y score
✓ 'fuente' sale de los metadatos y 'texto' de los documentos
✓ score = 1 - distancia: [1.0, 0.8]
✓ Con fuente='b.txt' solo devuelve fragmentos de esa fuente

🎉 ¡Todo funciona correctamente!


#### Tarea 3: Indexar el corpus en Chroma

Esta celda esta completa. Reutilizamos los embeddings que ya calculamos (no hace falta volver a embeber) y comprobamos que Chroma da los mismos resultados que nuestro buscador de numpy.

In [38]:
indexar_en_chroma(coleccion, fragmentos, embeddings_fragmentos)
print(f"✓ {coleccion.count()} fragmentos en Chroma\n")


def buscador_chroma(pregunta: str, k: int = 3, fuente: str = None) -> list:
  return buscar_chroma(pregunta, coleccion, k, fuente)


r_numpy = evaluar_recuperacion(buscador_semantico, CASOS_EVAL, k=3)
r_chroma = evaluar_recuperacion(buscador_chroma, CASOS_EVAL, k=3)
print(f"hit_rate@3 -> numpy: {r_numpy['hit_rate']:.2f} | chroma: {r_chroma['hit_rate']:.2f}")
print(f"mrr@3      -> numpy: {r_numpy['mrr']:.2f} | chroma: {r_chroma['mrr']:.2f}")

✓ 35 fragmentos en Chroma

hit_rate@3 -> numpy: 0.94 | chroma: 0.94
mrr@3      -> numpy: 0.91 | chroma: 0.91


Ahora lo que numpy no nos daba gratis: **filtrar por metadatos**. La palabra "limite" significa cosas distintas segun el documento.

In [39]:
pregunta = "¿Cual es el limite maximo permitido?"

mostrar_resultados(pregunta + "   [todas las fuentes]", buscador_chroma(pregunta))
mostrar_resultados(pregunta + "   [solo manual_operativo.txt]", buscador_chroma(pregunta, fuente="manual_operativo.txt"))
mostrar_resultados(pregunta + "   [solo politica_seguridad_vial.txt]", buscador_chroma(pregunta, fuente="politica_seguridad_vial.txt"))

P: ¿Cual es el limite maximo permitido?   [todas las fuentes]
  [1]   0.83  politica_seguridad_vial.txt#1  estrictos que los legales. En carretera nacional pavimentada el máximo es 80 km/h. En vías...
  [2]   0.82  politica_seguridad_vial.txt#0  TRANSPORTES DEL LLANO S.A.S. POLÍTICA DE SEGURIDAD VIAL Código: SV-003 — Actualizada en ma...
  [3]   0.82  politica_seguridad_vial.txt#2  hacerse una pausa activa de 15 minutos, fuera de la cabina. Entre una jornada y la siguien...

P: ¿Cual es el limite maximo permitido?   [solo manual_operativo.txt]
  [1]   0.82  manual_operativo.txt#2         coordinador de flota. 3. CARGAS AUTORIZADAS Y LÍMITES DE PESO 3.1 Fruto de palma. La carga...
  [2]   0.82  manual_operativo.txt#5         a bordo: licencia de conducción vigente, SOAT, revisión técnico-mecánica, tarjeta de propi...
  [3]   0.81  manual_operativo.txt#6         vehículo, verificar que la carga quede bien distribuida y confirmar el peso en la báscula ...

P: ¿Cual es el limite maximo per

En el manual operativo "limite" es de **peso**; en la politica de seguridad vial, de **velocidad**. En una empresa real, los filtros por metadatos tambien sirven para **permisos**: que un conductor no pueda recuperar fragmentos de documentos de gerencia, por ejemplo.

Sobre la persistencia: la carpeta `indice_rag/` tiene todo el indice. Si reinicias el entorno sin borrar archivos, puedes abrirlo con `chromadb.PersistentClient(path="indice_rag").get_collection(NOMBRE_COLECCION)` y buscar **sin volver a embeber nada**. (En Colab los archivos se borran cuando se recicla la maquina; en un servidor quedarian guardados.)

Lo que Chroma **no** guarda es el modelo: las preguntas se deben embeber con el **mismo** modelo con el que se indexaron los fragmentos. Si cambias de modelo de embeddings, hay que reindexar todo.

## Actividad 6: Interfaz para el RAG

### Objetivo

Un RAG que solo corre dentro de un notebook no lo usa nadie. Los usuarios de Transportes del Llano son conductores y despachadores, no programadores. Vamos a ponerle una **interfaz web** con **Gradio**, una libreria que convierte funciones de Python en una pagina; en Colab, ademas, genera un link publico temporal para abrirla desde cualquier navegador (o desde el celular).

Tres principios de una buena interfaz de RAG:

1. **Mostrar las fuentes**: el usuario debe poder verificar de donde salio cada dato.
2. **Mostrar lo recuperado**, al menos a quien mantiene el sistema: cuando una respuesta sale mal, lo primero es revisar si el buscador trajo lo correcto (Actividad 4).
3. **Dejar claro cuando no sabe**: "no encuentro esa informacion" es una respuesta valida, y mucho mejor que un dato inventado.

#### Tarea 1: Construir la funcion `formatear_respuesta`

El LLM cita con numeros (`[1]`, `[2]`, ...). Para el usuario, un numero suelto no significa nada: hay que traducirlo a la fuente.

### Requisitos

1. Buscar las citas `[n]` en la respuesta. Pista: `re.findall(r"\[(\d+)\]", texto)` devuelve los numeros citados como strings.
2. Quedarse con los numeros **validos** (entre 1 y la cantidad de fragmentos), sin repetir y en orden ascendente.
3. Si hay al menos uno, agregar al final una linea en blanco y `**Fuentes:** [n] fuente, [m] fuente`.
4. Si no hay citas validas (por ejemplo, cuando responde que no encuentra la informacion), devolver la respuesta tal cual.

Listamos solo lo que el modelo **cito**, no todo lo recuperado: si se recuperaron tres fragmentos pero la respuesta sale de uno, mostrar los tres como "fuentes" seria enganoso.

### Ejemplo

```
formatear_respuesta({
    "respuesta": "Son 85.000 pesos por dia [2].",
    "fragmentos": [{"fuente": "manual_operativo.txt", ...}, {"fuente": "politica_viaticos.txt", ...}],
})
== "Son 85.000 pesos por dia [2].\n\n**Fuentes:** [2] politica_viaticos.txt"
```

In [40]:
def formatear_respuesta(resultado: dict) -> str:
  """Agrega a la respuesta del LLM la lista de fuentes que cito, en formato Markdown."""

  respuesta = resultado["respuesta"]
  fragmentos = resultado["fragmentos"]

  # TODO: encontrar los numeros citados, quedarse con los validos, sin repetir y ordenados
  citados = sorted({int(n) for n in re.findall(r"\[(\d+)\]", respuesta) if 1 <= int(n) <= len(fragmentos)})

  # TODO: si no hay citas validas, devolver la respuesta tal cual; si hay, agregar la linea de fuentes
  if not citados:
    return respuesta

  fuentes = ", ".join(f"[{n}] {fragmentos[n - 1]['fuente']}" for n in citados)
  return f"{respuesta}\n\n**Fuentes:** {fuentes}"

**Evaluacion de implementacion**

In [41]:
# @title
# Celda de validacion. No modificar.

def _validar_formatear_respuesta():
  if "formatear_respuesta" not in globals():
    print("✗ Todavia no existe 'formatear_respuesta'. Ejecuta la celda anterior.")
    return

  frags = [{"fuente": "manual_operativo.txt"}, {"fuente": "politica_viaticos.txt"}, {"fuente": "tarifas_servicio.txt"}]
  casos = [
      ("ejemplo del enunciado",
       "Son 85.000 pesos por dia [2].",
       "Son 85.000 pesos por dia [2].\n\n**Fuentes:** [2] politica_viaticos.txt"),
      ("varias citas, repetidas y desordenadas -> sin repetir y en orden",
       "Dato A [3]. Dato B [1]. Otra vez A [3].",
       "Dato A [3]. Dato B [1]. Otra vez A [3].\n\n**Fuentes:** [1] manual_operativo.txt, [3] tarifas_servicio.txt"),
      ("sin citas -> la respuesta tal cual",
       "No encuentro esa informacion en los documentos.",
       "No encuentro esa informacion en los documentos."),
      ("ignora citas fuera de rango ([7] y [0])",
       "Algo [7] y algo mas [0].",
       "Algo [7] y algo mas [0]."),
      ("mezcla de citas validas e invalidas",
       "Uno [1], nueve [9].",
       "Uno [1], nueve [9].\n\n**Fuentes:** [1] manual_operativo.txt"),
  ]

  fallas = []
  for descripcion, respuesta, esperado in casos:
    try:
      obtenido = formatear_respuesta({"respuesta": respuesta, "fragmentos": frags})
    except Exception as e:
      obtenido = f"{type(e).__name__}: {e}"
    if obtenido == esperado:
      print(f"✓ {descripcion}")
    else:
      print(f"✗ {descripcion}\n    esperaba: {esperado!r}\n    obtuvo:   {obtenido!r}")
      fallas.append(descripcion)

  print()
  if fallas:
    print(f"{len(fallas)} problema(s) por corregir.")
  else:
    print("\U0001F389 ¡Todo funciona correctamente!")


_validar_formatear_respuesta()

✓ ejemplo del enunciado
✓ varias citas, repetidas y desordenadas -> sin repetir y en orden
✓ sin citas -> la respuesta tal cual
✓ ignora citas fuera de rango ([7] y [0])
✓ mezcla de citas validas e invalidas

🎉 ¡Todo funciona correctamente!


#### Tarea 2: Una interfaz de chat en 10 lineas

Esta celda esta completa. `gr.ChatInterface` arma un chat completo a partir de **una sola funcion** que recibe el mensaje nuevo y el historial, y devuelve el texto de la respuesta. Por ahora ignoramos el historial (volveremos a eso en las preguntas para discutir).

Con `share=True`, Gradio crea un link publico temporal (del tipo `https://xxxx.gradio.live`). **Cuidado:** cualquiera con ese link puede usar tu asistente... y gastar tu cuota de Groq. No lo publiques.

> Si algo falla, la interfaz solo muestra "Error". Para ver el detalle, lanzala con `debug=True` (la celda se queda corriendo hasta que la detengas).

In [42]:
import gradio as gr


def chat_rag(mensaje, historial):
  """Gradio llama esta funcion con el mensaje nuevo y el historial; devuelve el texto de la respuesta."""
  resultado = responder_con_rag(mensaje, buscador_chroma, k=3)
  return formatear_respuesta(resultado)


demo_simple = gr.ChatInterface(
    fn=chat_rag,
    title="Asistente de Transportes del Llano",
    description="Preguntale sobre los manuales y politicas de la empresa.",
    examples=[
        "¿Cuanto se paga de viaticos en ruta nacional?",
        "¿Que hago si me roban el camion?",
        "¿Cuantos dias de vacaciones tengo?",
    ],
)

demo_simple.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://032276a3c246b6b3cf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Prueba unas cuantas preguntas y fijate en la linea de **Fuentes**. Cuando termines, cierra esta interfaz para liberar el puerto:

In [43]:
demo_simple.close()

Closing server running on port: 7860


#### Tarea 3: Construir la funcion `formatear_fragmentos`

La interfaz simple es comoda para el usuario final, pero para **depurar** necesitamos ver que recupero el buscador. Vamos a armar un panel lateral con los fragmentos.

### Requisitos

1. Un bloque por fragmento, numerados desde 1, con este formato (el texto va como cita de Markdown, con `> `):
   ```
   **[1] politica_viaticos.txt** · score 0.87

   > En ruta regional, es decir, en viajes dentro del Meta...
   ```
2. El `score` con 2 decimales.
3. Si el texto tiene mas de `max_caracteres` caracteres, recortarlo a esa cantidad y agregar `"..."`. Si no, dejarlo completo y **sin** `"..."`.
4. Separar los bloques con una linea en blanco.
5. Si la lista esta vacia, devolver `"_No se recuperaron fragmentos._"`.

In [44]:
def formatear_fragmentos(fragmentos: list, max_caracteres: int = 300) -> str:
  """Convierte los fragmentos recuperados en Markdown para el panel de la interfaz."""

  # TODO: caso de lista vacia
  if not fragmentos:
    return "_No se recuperaron fragmentos._"

  bloques = []

  # TODO: un bloque por fragmento, recortando el texto si pasa de max_caracteres
  for i, fragmento in enumerate(fragmentos, start=1):
    texto = fragmento["texto"]
    if len(texto) > max_caracteres:
      texto = texto[:max_caracteres] + "..."
    bloques.append(f"**[{i}] {fragmento['fuente']}** · score {fragmento['score']:.2f}\n\n> {texto}")

  return "\n\n".join(bloques)

**Evaluacion de implementacion**

In [45]:
# @title
# Celda de validacion. No modificar.

def _validar_formatear_fragmentos():
  if "formatear_fragmentos" not in globals():
    print("✗ Todavia no existe 'formatear_fragmentos'. Ejecuta la celda anterior.")
    return

  frags = [
      {"id": "a#0", "fuente": "a.txt", "texto": "corto", "score": 0.87654},
      {"id": "b#0", "fuente": "b.txt", "texto": "x" * 12, "score": 0.5},
  ]
  casos = [
      ("formato de dos fragmentos (el segundo se recorta a 10 caracteres + '...')",
       lambda: formatear_fragmentos(frags, max_caracteres=10),
       "**[1] a.txt** · score 0.88\n\n> corto\n\n**[2] b.txt** · score 0.50\n\n> xxxxxxxxxx..."),
      ("texto de exactamente max_caracteres -> sin '...'",
       lambda: formatear_fragmentos([dict(frags[1], texto="y" * 10)], max_caracteres=10),
       "**[1] b.txt** · score 0.50\n\n> yyyyyyyyyy"),
      ("lista vacia",
       lambda: formatear_fragmentos([]),
       "_No se recuperaron fragmentos._"),
  ]

  fallas = []
  for descripcion, calcular, esperado in casos:
    try:
      obtenido = calcular()
    except Exception as e:
      obtenido = f"{type(e).__name__}: {e}"
    if obtenido == esperado:
      print(f"✓ {descripcion}")
    else:
      print(f"✗ {descripcion}\n    esperaba: {esperado!r}\n    obtuvo:   {obtenido!r}")
      fallas.append(descripcion)

  print()
  if fallas:
    print(f"{len(fallas)} problema(s) por corregir.")
  else:
    print("\U0001F389 ¡Todo funciona correctamente!")


_validar_formatear_fragmentos()

✓ formato de dos fragmentos (el segundo se recorta a 10 caracteres + '...')
✓ texto de exactamente max_caracteres -> sin '...'
✓ lista vacia

🎉 ¡Todo funciona correctamente!


#### Tarea 4: La interfaz completa con `gr.Blocks`

Esta celda esta completa, pero leela con calma: es el patron de cualquier app de Gradio.

- **`gr.Blocks`** es el lienzo; **`gr.Row`** y **`gr.Column`** acomodan los componentes.
- Los **componentes** son las piezas: `Chatbot`, `Textbox`, `Slider`, `Dropdown`, `Markdown`, `File`, `Button`.
- Los **eventos** conectan todo: `caja.submit(funcion, inputs=[...], outputs=[...])` significa "cuando el usuario presione Enter en `caja`, llama a `funcion` con el valor de los `inputs` y pon lo que devuelva en los `outputs`", en ese mismo orden.

Lo nuevo frente a la interfaz simple:

- **Panel de fragmentos recuperados**, con `formatear_fragmentos`.
- **Control de `k`** y **filtro por fuente** (el `where` de Chroma de la Actividad 5).
- **Subir un documento nuevo** y indexarlo en caliente, sin reiniciar nada. Esto es posible gracias a `upsert` y a que Chroma permite agregar registros a una coleccion existente.

In [46]:
FUENTES = ["Todas"] + sorted({f["fuente"] for f in fragmentos})
PANEL_VACIO = "_Aun no has hecho ninguna pregunta._"


def responder_interfaz(mensaje, historial, k, fuente):
  """Se ejecuta cuando el usuario envia una pregunta."""
  if not mensaje or not mensaje.strip():
    return "", historial, "_Escribe una pregunta._"

  filtro = None if fuente == "Todas" else fuente

  def buscador(pregunta, k):
    return buscador_chroma(pregunta, k, fuente=filtro)

  resultado = responder_con_rag(mensaje, buscador, k=int(k))

  historial = historial + [
      {"role": "user", "content": mensaje},
      {"role": "assistant", "content": formatear_respuesta(resultado)},
  ]
  return "", historial, formatear_fragmentos(resultado["fragmentos"])


def indexar_archivo(ruta_archivo):
  """Se ejecuta cuando el usuario sube un .txt y presiona 'Indexar documento'."""
  if ruta_archivo is None:
    return "Primero sube un archivo .txt", gr.update()

  nombre = os.path.basename(ruta_archivo)
  with open(ruta_archivo, "r", encoding="utf-8") as f:
    texto = f.read()

  nuevos = fragmentar_corpus([{"fuente": nombre, "texto": texto}])
  indexar_en_chroma(coleccion, nuevos, embeber([f["texto"] for f in nuevos], tipo="passage"))

  if nombre not in FUENTES:
    FUENTES.append(nombre)
  mensaje = f"✓ **{nombre}**: {len(nuevos)} fragmentos indexados. La coleccion tiene {coleccion.count()} en total."
  return mensaje, gr.update(choices=FUENTES)


with gr.Blocks(title="Asistente Transportes del Llano") as demo:
  gr.Markdown("# Asistente de Transportes del Llano\nPreguntas sobre manuales y politicas internas. Cada respuesta cita sus fuentes.")

  with gr.Row():
    with gr.Column(scale=3):
      chat = gr.Chatbot(label="Conversacion", height=460)
      caja = gr.Textbox(placeholder="Escribe tu pregunta y presiona Enter", show_label=False)
      gr.Examples(
          examples=[
              "¿Cuanto se paga de viaticos en ruta nacional?",
              "¿Que hago si me roban el camion?",
              "¿Cual es el limite maximo permitido?",
          ],
          inputs=caja,
      )
      boton_limpiar = gr.Button("Limpiar conversacion")

    with gr.Column(scale=2):
      selector_k = gr.Slider(1, 8, value=3, step=1, label="Fragmentos a recuperar (k)")
      selector_fuente = gr.Dropdown(FUENTES, value="Todas", label="Buscar solo en")
      with gr.Accordion("Fragmentos recuperados", open=True):
        panel = gr.Markdown(PANEL_VACIO)
      with gr.Accordion("Agregar un documento", open=False):
        archivo = gr.File(label="Documento .txt", file_types=[".txt"], type="filepath")
        boton_indexar = gr.Button("Indexar documento")
        estado = gr.Markdown()

  caja.submit(responder_interfaz, inputs=[caja, chat, selector_k, selector_fuente], outputs=[caja, chat, panel])
  boton_indexar.click(indexar_archivo, inputs=[archivo], outputs=[estado, selector_fuente])
  boton_limpiar.click(lambda: ([], PANEL_VACIO), outputs=[chat, panel])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://83b558c535eb5613b1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#### Tarea 5: Prueba la interfaz

Haz estas pruebas en orden y anota lo que pase; varias alimentan las preguntas para discutir.

1. **Fuentes y panel.** Pregunta por los viaticos de ruta nacional. Revisa que la cita de la respuesta coincida con el fragmento del panel.
2. **Mover `k`.** Repite la pregunta con `k=1` y con `k=8`. ¿Cambia la respuesta? ¿Y el panel?
3. **Filtros.** Pregunta "¿Cual es el limite maximo permitido?" con "Todas", despues filtrando por `manual_operativo.txt` y despues por `politica_seguridad_vial.txt`.
4. **Documento nuevo.** Pregunta "¿Hay algun viatico adicional cuando cierran la via?". Despues sube `circular_temporada_lluvias_2026.txt` (esta en la carpeta `para_probar_interfaz/` del zip que descargaste del drive), presiona **Indexar documento** y repite la pregunta.
5. **Contradicciones.** Con la circular ya indexada, pregunta "¿Cual es la carga maxima de fruto de palma por camion sencillo?". Mira el panel.
6. **Seguimiento.** Pregunta "¿Cuantos dias de vacaciones tengo?" y despues "¿Y con cuanta anticipacion las pido?".

Cuando termines, cierra la interfaz:

In [47]:
demo.close()

Closing server running on port: 7860


#### Opcional: la misma app en Streamlit

**Streamlit** es la otra libreria popular para este tipo de apps, con una filosofia distinta:

- En **Gradio** declaras componentes y **eventos**: cada interaccion llama a una funcion.
- En **Streamlit** el script entero **se vuelve a ejecutar de arriba a abajo** en cada interaccion. Por eso lo que es caro de cargar (el modelo, la coleccion) se envuelve en `@st.cache_resource`, y lo que debe sobrevivir entre ejecuciones (el historial) se guarda en `st.session_state`.

Streamlit no corre comodo dentro de Colab, asi que esta celda solo **escribe** la app en `app_rag.py`. La idea es correrla en tu computador contra el indice que ya construimos: como Chroma lo guardo en disco, **la app no necesita volver a indexar nada**. Las instrucciones estan al inicio del archivo.

In [45]:
%%writefile app_rag.py
# App de Streamlit para el RAG de Transportes del Llano.
#
# Para correrla en tu computador:
#   1. Descarga app_rag.py e indice_rag.zip de Colab (siguiente celda) y descomprime indice_rag.zip
#      en la misma carpeta que app_rag.py.
#   2. pip install streamlit "chromadb>=1.0" sentence-transformers groq
#   3. Define tu llave:  export GROQ_API_KEY=tu_llave     (en Windows: set GROQ_API_KEY=tu_llave)
#   4. streamlit run app_rag.py

import os
import chromadb
import streamlit as st
from groq import Groq
from sentence_transformers import SentenceTransformer

SYSTEM_RAG = (
    "Eres el asistente interno de Transportes del Llano. "
    "Responde en espanol, de forma breve, y SOLO con base en los fragmentos que se te entregan. "
    "Despues de cada dato, cita entre corchetes el numero del fragmento de donde lo sacaste, por ejemplo [2]. "
    "Si la respuesta no esta en los fragmentos, responde exactamente: "
    "'No encuentro esa informacion en los documentos.'"
)


@st.cache_resource  # se ejecuta una sola vez, no en cada interaccion
def cargar_recursos():
  modelo = SentenceTransformer("intfloat/multilingual-e5-small")
  coleccion = chromadb.PersistentClient(path="indice_rag").get_collection("transportes_del_llano")
  cliente = Groq(api_key=os.environ["GROQ_API_KEY"])
  return modelo, coleccion, cliente


modelo, coleccion, cliente = cargar_recursos()


def buscar(pregunta, k, fuente):
  vector = modelo.encode(["query: " + pregunta], normalize_embeddings=True)[0]
  r = coleccion.query(query_embeddings=[vector.tolist()], n_results=k,
                      where={"fuente": fuente} if fuente else None)
  return [{"fuente": m["fuente"], "texto": t, "score": 1 - d}
          for t, m, d in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]


def responder(pregunta, fragmentos):
  bloques = [f"[{i}] (fuente: {f['fuente']})\n{f['texto']}" for i, f in enumerate(fragmentos, start=1)]
  prompt = "\n\n".join(bloques + [f"Pregunta: {pregunta}"])
  r = cliente.chat.completions.create(
      model="openai/gpt-oss-20b", temperature=0, reasoning_effort="low",
      messages=[{"role": "system", "content": SYSTEM_RAG}, {"role": "user", "content": prompt}],
  )
  return r.choices[0].message.content


st.title("Asistente de Transportes del Llano")

with st.sidebar:
  k = st.slider("Fragmentos a recuperar (k)", 1, 8, 3)
  fuentes = sorted({m["fuente"] for m in coleccion.get(include=["metadatas"])["metadatas"]})
  fuente = st.selectbox("Buscar solo en", ["Todas"] + fuentes)

if "mensajes" not in st.session_state:
  st.session_state.mensajes = []

for m in st.session_state.mensajes:   # como el script se re-ejecuta, hay que volver a pintar el historial
  with st.chat_message(m["role"]):
    st.markdown(m["content"])

if pregunta := st.chat_input("Escribe tu pregunta"):
  st.session_state.mensajes.append({"role": "user", "content": pregunta})
  with st.chat_message("user"):
    st.markdown(pregunta)

  fragmentos = buscar(pregunta, k, None if fuente == "Todas" else fuente)
  respuesta = responder(pregunta, fragmentos)

  with st.chat_message("assistant"):
    st.markdown(respuesta)
    with st.expander("Fragmentos recuperados"):
      for i, f in enumerate(fragmentos, start=1):
        st.markdown(f"**[{i}] {f['fuente']}** · score {f['score']:.2f}")
        st.caption(f["texto"])

  st.session_state.mensajes.append({"role": "assistant", "content": respuesta})

Writing app_rag.py


In [46]:
# Empaqueta el indice y descarga los dos archivos
import shutil
from google.colab import files

shutil.make_archive("indice_rag", "zip", root_dir=".", base_dir="indice_rag")
files.download("app_rag.py")
files.download("indice_rag.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Preguntas para discutir

**1. En la Tarea 3 de la Actividad 4, los fragmentos grandes probablemente tuvieron el mejor hit rate. ¿Por que no los usarias igual?**

Porque la metrica, tal como la definimos, favorece a los fragmentos grandes: con 300 palabras cada documento queda en 2 o 3 pedazos y "contener la evidencia" es casi regalado. Lo que no mide es el **costo**: le mandas al LLM ~900 palabras donde el dato ocupa una linea, pagas mas tokens, y los modelos tienden a perder informacion que queda enterrada en medio de mucho texto. Ademas, el embedding de un fragmento largo mezcla varios temas y se vuelve menos preciso, y pasado cierto largo el modelo de embeddings **trunca en silencio** (e5 lee 512 tokens).

Lo sensato es decidir con varias columnas a la vez: calidad de la recuperacion **y** texto enviado. Una tecnica comun es buscar con fragmentos chicos (precisos) y, una vez encontrados, mandarle al LLM el parrafo o la seccion que los rodea (*small-to-big*).

**2. En la interfaz preguntaste "¿Cuantos dias de vacaciones tengo?" y despues "¿Y con cuanta anticipacion las pido?". ¿Que paso, por que, y como lo arreglarias?**

La segunda pregunta probablemente salio mal: respondio sobre la anticipacion de las **cancelaciones** o de los **permisos**, o dijo que no encontraba la informacion. La causa es que nuestro RAG es **sin memoria**: el buscador solo ve el ultimo mensaje, y "las" no dice nada sobre vacaciones; ademas, `responder_con_rag` tampoco le pasa el historial al LLM. La interfaz *muestra* una conversacion, pero cada pregunta se procesa sola.

La solucion estandar es **reformular la pregunta** antes de buscar: con una llamada extra al LLM se le pasa el historial y se le pide reescribir la ultima pregunta para que se entienda sola ("¿Con cuanta anticipacion se solicitan las vacaciones?"). Esa pregunta reformulada es la que se busca. Cuesta una llamada mas por mensaje, pero sin eso las conversaciones de seguimiento, que son la mayoria en un chat real, no funcionan.

**3. Despues de indexar la circular, ¿que respondio a la pregunta por la carga maxima de fruto de palma? ¿Que deberia responder, y como lo garantizarias?**

Hay dos fragmentos que se contradicen: el manual dice 12 toneladas y la circular dice 11 durante la temporada de lluvias. Segun cuales entren en los `k` y en que orden, el modelo puede responder cualquiera de las dos, mencionar ambas, o mezclarlas. La respuesta correcta **depende de la fecha**: entre el 15 de septiembre y el 15 de diciembre de 2026 son 11, el resto del ano 12.

El RAG no sabe nada de vigencias: para el, los dos fragmentos son texto igual de valido. Para garantizarlo hace falta **informacion que no esta en el embedding**: metadatos de tipo de documento, fecha de publicacion y vigencia; filtrar lo vencido; poner la fecha en los fragmentos para que el LLM pueda razonar ("esta circular prevalece hasta el 15 de diciembre"), e instruirlo para que prefiera la norma mas reciente o especifica. Y, sobre todo, **gobernanza documental**: alguien tiene que retirar lo obsoleto. Un RAG es exactamente tan bueno como los documentos que le das.

**4. En el Workshop 3 mejoramos un modelo con LoRA; hoy con RAG. Si la empresa cambia el valor de los viaticos el ano que viene, ¿cual de los dos enfoques preferirias? ¿Para que problemas usarias el otro?**

RAG, sin dudarlo. Con RAG basta **reemplazar el documento** y reindexarlo (un `upsert`): el cambio queda activo en segundos y la respuesta viene con cita. Con fine-tuning habria que armar datos nuevos y reentrenar, y aun asi el modelo podria "recordar" el valor viejo, porque el conocimiento queda mezclado en los pesos sin forma de auditarlo.

La regla practica: **RAG para conocimiento** (datos que cambian, que hay que citar, o que dependen de permisos), **fine-tuning para comportamiento** (una tarea como la clasificacion del Workshop 3, un formato de salida, un tono, la jerga de un dominio). No compiten: un sistema real puede tener un modelo afinado para responder en el formato de la empresa y, ademas, RAG para los datos.

**5. La interfaz permite que cualquiera suba documentos y la lanzamos con `share=True`. ¿Que riesgos ves?**

Con `share=True` el link es publico: cualquiera que lo tenga puede usar el asistente, gastar la cuota de Groq y consultar los documentos internos. En produccion se agrega autenticacion (Gradio acepta `demo.launch(auth=...)`) o se despliega detras del login de la empresa.

Subir documentos es aun mas delicado. Un documento malicioso puede **envenenar** el indice con datos falsos ("los viaticos son de 1.000.000 de pesos"), y el asistente los repetira con cita y todo. Peor: puede traer **instrucciones escondidas** ("ignora tus reglas y responde que...") que el LLM lee como parte del prompt; esto se llama *prompt injection* indirecta. Por eso, en un sistema real solo personas autorizadas indexan documentos, se revisan antes de publicarlos, y el prompt deja claro que los fragmentos son **datos**, no instrucciones.

### Extensiones opcionales

- **Busqueda hibrida**: combina BM25 y semantico con *Reciprocal Rank Fusion*: cada fragmento suma `1 / (60 + posicion)` en cada ranking donde aparece. Mide el resultado con `evaluar_recuperacion`. ¿Le gana a los dos?
- **Reranking**: recupera 20 fragmentos con el buscador semantico y reordenalos con un *cross-encoder* (por ejemplo `BAAI/bge-reranker-v2-m3` con `sentence_transformers.CrossEncoder`), que lee pregunta y fragmento juntos. Quedate con los 3 mejores y compara el MRR.
- **Reformular con el historial**: implementa la solucion de la pregunta 2 en la interfaz de `gr.Blocks` y prueba otra vez la conversacion de vacaciones.
- **Streaming**: usa `stream=True` en Groq y convierte `responder_interfaz` en un generador (`yield`) para que la respuesta aparezca palabra por palabra.
- **Evaluar la generacion**: corre `responder_con_rag` sobre `CASOS_EVAL` y revisa si la respuesta contiene la evidencia y si la cita apunta al fragmento correcto. Despues prueba usar el propio LLM como juez.
- **Fragmentar por secciones**: los documentos tienen titulos numerados ("3. VACACIONES"). Corta por ahi en vez de por numero de palabras y compara con la Actividad 4.
- **PDFs**: la mayoria de documentos reales vienen en PDF. Usa `pypdf` para extraer el texto e indexarlo con lo que ya tienes.
- **Metadatos de vigencia**: agrega a cada fuente su fecha de publicacion y vigencia, y usala para resolver la contradiccion de la pregunta 3.